# Reproduction notebook: 36F_Exchange_full8_segmoe_oof_reranker_predeclared_small_backend_safe_v2

This notebook is retained as an executable provenance record for the anonymous supplementary package. Saved outputs and internal development notes have been removed.


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import os
import subprocess
import sys

# ---------------------------------------------------------------------
# 1. Resolve / clone official Seg-MoE repository FIRST.
# ---------------------------------------------------------------------
SEGMOE_REPO_CANDIDATES = []

if os.environ.get("SEGMOE_REPO"):
    SEGMOE_REPO_CANDIDATES.append(
        Path(os.environ["SEGMOE_REPO"])
    )

SEGMOE_REPO_CANDIDATES += [
    Path("/data/segmoe_forecast"),
    Path("/data/Time-Series-Library_v2/segmoe_forecast"),
    Path("/data/Time-Series-Library/segmoe_forecast"),
    Path("/code/segmoe_forecast"),
    Path("/workspace/segmoe_forecast"),
    Path.cwd() / "segmoe_forecast",
    Path.cwd(),
]

EXPECTED_SEGMOE_FILE = Path(
    "segmoe_forecast/model/TSFT.py"
)

SEGMOE_REPO = next(
    (
        p.resolve()
        for p in SEGMOE_REPO_CANDIDATES
        if (p / EXPECTED_SEGMOE_FILE).is_file()
    ),
    None,
)

if SEGMOE_REPO is None:
    fallback = Path("/data/segmoe_forecast")
    fallback.parent.mkdir(parents=True, exist_ok=True)

    print(
        "Seg-MoE repository not found locally. "
        "Cloning official repository ->",
        fallback,
    )

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/evortigosa/segmoe_forecast.git",
            str(fallback),
        ],
        check=True,
    )

    SEGMOE_REPO = fallback.resolve()

# CRITICAL: add repository root before importing segmoe_forecast.
if str(SEGMOE_REPO) not in sys.path:
    sys.path.insert(0, str(SEGMOE_REPO))

importlib.invalidate_caches()

# ---------------------------------------------------------------------
# 2. Check required non-torch dependencies.
#    Do NOT modify the user's PyTorch installation.
# ---------------------------------------------------------------------
required_modules = {
    "timm": "timm==1.0.24",
    "einops": "einops==0.8.2",
    "datasetsforecast": "datasetsforecast==1.0.0",
}

missing = [
    pip_name
    for module_name, pip_name in required_modules.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    raise ModuleNotFoundError(
        "Missing Seg-MoE dependencies: "
        + ", ".join(missing)
        + "\nInstall them in the current kernel environment, e.g.\n"
        + f"{sys.executable} -m pip install "
        + " ".join(missing)
        + "\nThen restart this cell. "
        + "Do NOT reinstall/downgrade torch."
    )

# ---------------------------------------------------------------------
# 3. Imports only after sys.path + dependency checks.
# ---------------------------------------------------------------------
import torch
import timm
import einops
import datasetsforecast

from segmoe_forecast.model import TSFTransformer
from segmoe_forecast.model.Config import SmallConfig

try:
    import segmoe_forecast
    SEGMOE_PACKAGE_PATH = Path(segmoe_forecast.__file__).resolve()
except Exception:
    SEGMOE_PACKAGE_PATH = None

try:
    SEGMOE_COMMIT = subprocess.check_output(
        [
            "git",
            "-C",
            str(SEGMOE_REPO),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    ).strip()
except Exception:
    SEGMOE_COMMIT = "unknown"

print("=" * 90)
print("36F ENVIRONMENT PREFLIGHT")
print("=" * 90)
print("python:", sys.executable)
print("torch:", torch.__version__)
print("timm:", timm.__version__)
print("einops:", einops.__version__)
print("datasetsforecast:", datasetsforecast.__version__)
print("Seg-MoE repo:", SEGMOE_REPO)
print("Seg-MoE package:", SEGMOE_PACKAGE_PATH)
print("Seg-MoE commit:", SEGMOE_COMMIT)
print("PASS: Seg-MoE imports successful.")


In [ ]:
from pathlib import Path
from contextlib import nullcontext
from dataclasses import replace, asdict
from torch.utils.data import Dataset, DataLoader

import gc
import importlib
import inspect
import math
import os
import random
import shutil
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 560)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

DATASET = "Exchange"
CURRENT_BACKBONE = "SegMoE"
HORIZONS = [96, 192, 336, 720]

# ---------------------------------------------------------------------
# Existing retrieval protocol: unchanged.
# ---------------------------------------------------------------------
RET_SEQ_LEN = 96
TOP_K = 10
MEMORY_STRIDE = 8
OOF_ANCHOR_STRIDE = 4

FOLDS = [
    (0.55, 0.70),
    (0.70, 0.85),
    (0.85, 1.00),
]

REP_PATCH_LEN = 16
REP_PATCH_STRIDE = 16
REP_D_MODEL = 64
REP_N_HEADS = 4
REP_LAYERS = 2
REP_D_FF = 128
REP_DROPOUT = 0.1
REP_DIM = 64

REP_NUM_PATCHES = (
    1
    + (
        RET_SEQ_LEN
        - REP_PATCH_LEN
    )
    // REP_PATCH_STRIDE
)

GATE_DIM = 26
GATE_LR = 1e-3
GATE_WD = 1e-4
GATE_BATCH = 8192
GATE_MAX_EPOCHS = 50
GATE_PATIENCE = 7

ALPHA_GRID = np.round(
    np.arange(
        0.0,
        1.0001,
        0.1,
    ),
    10,
)

LAMBDA_GRID = np.array(
    [0.0, 0.25, 0.50, 0.75, 1.00],
    dtype=np.float32,
)

TARGET_RETRIEVAL_PAIRS = 672
EPS = 1e-8
RETRIEVER_USE_AMP = torch.cuda.is_available()

CROSSFIT_SEED = 36000
BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_BLOCK_LEN = 24
BOOTSTRAP_SEED = 360000

RESUME = True
FORCE = False

# ---------------------------------------------------------------------
# Official Seg-MoE Exchange-small recipe.
# ---------------------------------------------------------------------
SEGMOE_BLOCK_SIZE = 512
SEGMOE_PATCH_WIDTH = 8
SEGMOE_WIDTH_FACTOR = 4
SEGMOE_STEP = int(
    SEGMOE_PATCH_WIDTH
    * SEGMOE_WIDTH_FACTOR
)
SEGMOE_LABEL_LEN = (
    SEGMOE_BLOCK_SIZE
    - SEGMOE_STEP
)

SEGMOE_SEGMENT_SIZE = [4, 5, 5, 4]
SEGMOE_EPOCHS = 20
SEGMOE_MAX_LR = 3.2e-4
SEGMOE_MIN_LR = 1.2e-4
SEGMOE_WEIGHT_DECAY = 1e-4
SEGMOE_WARMUP = 0.1
SEGMOE_HUBER_DELTA = 2.0
SEGMOE_BAL_ALPHA = 0.02
SEGMOE_PATIENCE = 5
SEGMOE_STOP_MIN = 1e-6
SEGMOE_SEED = 55

# Official Exchange-small batch size.
# Override only if needed for GPU memory:
#   export SEGMOE_BATCH_SIZE=128
SEGMOE_BATCH_SIZE = int(
    os.environ.get(
        "SEGMOE_BATCH_SIZE",
        "256",
    )
)

SEGMOE_CLIP_GRAD = 1.0
SEGMOE_MODEL_SIZE = "small"

SEGMOE_BF16_REQUESTED = (
    os.environ.get("SEGMOE_USE_BF16", "1") == "1"
)

SEGMOE_BF16_SUPPORTED = bool(
    torch.cuda.is_available()
    and getattr(
        torch.cuda,
        "is_bf16_supported",
        lambda: False,
    )()
)

# Official Exchange-small requests BF16. Smoke test may fall back to FP32.
SEGMOE_BF16 = bool(
    SEGMOE_BF16_REQUESTED
    and SEGMOE_BF16_SUPPORTED
)

SEGMOE_PRECISION_MODE = (
    "BF16"
    if SEGMOE_BF16
    else "FP32"
)

# ---------------------------------------------------------------------
# Backend-stability policy for v4.
#
# Keep the Seg-MoE architecture/objective/hyperparameters unchanged,
# but avoid CUDA fused optimizer and Flash / memory-efficient SDPA
# backends in the current torch 2.12.x environment.
# ---------------------------------------------------------------------
SEGMOE_USE_FUSED_ADAMW = False
SEGMOE_DISABLE_FLASH_SDPA = True
SEGMOE_DISABLE_MEM_EFFICIENT_SDPA = True
SEGMOE_ENABLE_MATH_SDPA = True

SEGMOE_STABILITY_STEPS = int(
    os.environ.get(
        "SEGMOE_STABILITY_STEPS",
        "50",
    )
)

# Exchange has 21 channels; conservative vectorized direct inference.
DIRECT_ANCHOR_BLOCK = {
    "SegMoE": max(
        1,
        min(
            32,
            SEGMOE_BATCH_SIZE,
        ),
    )
}

EXPECTED_FULL_CHANNELS = 8

STRONG_FORECASTER_ROOT = Path(
    "/data/dataset/strong_forecaster"
)

EXP36F_ROOT = (
    STRONG_FORECASTER_ROOT
    / "exchange_full8_segmoe_oof_reranker"
)
EXP36F_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

ROOT = EXP36F_ROOT

SHARED_MEMORY_EMB_DIR = (
    EXP36F_ROOT
    / "memory_embeddings"
)
SHARED_MEMORY_EMB_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ARTIFACT_DIR = (
    EXP36F_ROOT
    / "artifacts"
)
ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DIRS = {
    "gate": EXP36F_ROOT / "gate",
    "history": EXP36F_ROOT / "history",
}
for _p in DIRS.values():
    _p.mkdir(
        parents=True,
        exist_ok=True,
    )

SUMMARY_36F = (
    EXP36F_ROOT
    / "summary.csv"
)
BOOTSTRAP_36F = (
    EXP36F_ROOT
    / "bootstrap.csv"
)

print("Device:", DEVICE)
print("BF16 requested:", SEGMOE_BF16_REQUESTED)
print("BF16 supported:", SEGMOE_BF16_SUPPORTED)
print("36F training precision:", SEGMOE_PRECISION_MODE)
print("Exchange Seg-MoE recipe source: predeclared ETTh1-small surrogate; no official Exchange recipe")
print("Fused AdamW:", SEGMOE_USE_FUSED_ADAMW)
print("Flash SDPA disabled:", SEGMOE_DISABLE_FLASH_SDPA)
print("Memory-efficient SDPA disabled:", SEGMOE_DISABLE_MEM_EFFICIENT_SDPA)
print("Math SDPA enabled:", SEGMOE_ENABLE_MATH_SDPA)
print("Stability steps:", SEGMOE_STABILITY_STEPS)
print("Seg-MoE model size:", SEGMOE_MODEL_SIZE)
print("Seg-MoE batch size:", SEGMOE_BATCH_SIZE)
print("Seg-MoE clip grad:", SEGMOE_CLIP_GRAD)
print("Direct evaluation anchor block:", DIRECT_ANCHOR_BLOCK["SegMoE"])
print("Output:", EXP36F_ROOT)


In [ ]:
def configure_segmoe_safe_backends():
    if not torch.cuda.is_available():
        print("CUDA unavailable: backend configuration skipped.")
        return

    if hasattr(torch.backends.cuda, "enable_flash_sdp"):
        torch.backends.cuda.enable_flash_sdp(
            not SEGMOE_DISABLE_FLASH_SDPA
        )

    if hasattr(torch.backends.cuda, "enable_mem_efficient_sdp"):
        torch.backends.cuda.enable_mem_efficient_sdp(
            not SEGMOE_DISABLE_MEM_EFFICIENT_SDPA
        )

    if hasattr(torch.backends.cuda, "enable_math_sdp"):
        torch.backends.cuda.enable_math_sdp(
            SEGMOE_ENABLE_MATH_SDPA
        )

    flash_enabled = (
        torch.backends.cuda.flash_sdp_enabled()
        if hasattr(torch.backends.cuda, "flash_sdp_enabled")
        else None
    )

    mem_enabled = (
        torch.backends.cuda.mem_efficient_sdp_enabled()
        if hasattr(torch.backends.cuda, "mem_efficient_sdp_enabled")
        else None
    )

    math_enabled = (
        torch.backends.cuda.math_sdp_enabled()
        if hasattr(torch.backends.cuda, "math_sdp_enabled")
        else None
    )

    print("=" * 90)
    print("36F SAFE CUDA BACKENDS")
    print("=" * 90)
    print("Flash SDPA enabled:", flash_enabled)
    print("Memory-efficient SDPA enabled:", mem_enabled)
    print("Math SDPA enabled:", math_enabled)

    if flash_enabled is True:
        raise RuntimeError(
            "Flash SDPA is still enabled; v4 requires it OFF."
        )

    if mem_enabled is True:
        raise RuntimeError(
            "Memory-efficient SDPA is still enabled; v4 requires it OFF."
        )

    if math_enabled is False:
        raise RuntimeError(
            "Math SDPA is disabled; v4 requires it ON."
        )

    print("PASS: safe SDPA backend configuration.")


configure_segmoe_safe_backends()


In [ ]:
EXCHANGE_CANDIDATES = [
    Path("/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
    Path("/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv"),
    Path("/data/pcw_workspace/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
    Path("/code/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
    Path("/data/dataset/exchange_rate/exchange_rate.csv"),
    Path("/data/dataset/exchange_rate.csv"),
]

EXCHANGE_PATH = next(
    (p for p in EXCHANGE_CANDIDATES if p.is_file()),
    None,
)

if EXCHANGE_PATH is None:
    print("Attempted Exchange paths:")
    for p in EXCHANGE_CANDIDATES:
        print(" -", p)
    raise FileNotFoundError("Could not find exchange_rate.csv.")


def load_exchange_csv(path):
    df = pd.read_csv(path)

    for col in list(df.columns):
        if str(col).lower() in {
            "date",
            "datetime",
            "timestamp",
            "time",
        }:
            df = df.drop(columns=[col])

    numeric = df.apply(pd.to_numeric, errors="coerce")
    numeric = numeric.dropna(axis=1, how="all")
    numeric = (
        numeric
        .replace([np.inf, -np.inf], np.nan)
        .interpolate(axis=0, limit_direction="both")
        .ffill()
        .bfill()
    )
    return numeric


raw_df = load_exchange_csv(EXCHANGE_PATH)
raw = raw_df.to_numpy(dtype=np.float32)
n_time, n_channels = raw.shape

if n_time != 7588:
    raise ValueError(
        f"Standard Exchange benchmark expects 7,588 rows, loaded {n_time}."
    )

if n_channels != EXPECTED_FULL_CHANNELS:
    raise ValueError(
        f"36F expects exactly {EXPECTED_FULL_CHANNELS} numeric channels, "
        f"loaded {n_channels}."
    )

# Same 70/10/20 chronological split used by the project's Exchange protocol.
train_end = int(0.70 * n_time)
val_end = int(0.80 * n_time)
test_end = n_time

assert train_end == 5311, train_end
assert val_end == 6070, val_end
assert test_end == 7588, test_end

train_mean = raw[:train_end].mean(axis=0).astype(np.float32)
train_std = raw[:train_end].std(axis=0, ddof=0).astype(np.float32)

degenerate = np.where(train_std <= 1e-6)[0]

if len(degenerate):
    raise ValueError(
        "Exchange contains train-degenerate channels: "
        f"{degenerate.tolist()}"
    )

z_full = (
    (raw - train_mean[None, :])
    / train_std[None, :]
).astype(np.float32)

marks = np.zeros((n_time, 0), dtype=np.float32)

DATA = {
    "Exchange": {
        "name": "Exchange",
        "path": EXCHANGE_PATH,
        "raw": raw,
        "z": z_full,
        "marks": marks,
        "n_channels": n_channels,
        "raw_channel_indices": np.arange(n_channels, dtype=np.int64),
        "train_end": train_end,
        "val_end": val_end,
        "test_end": test_end,
    }
}

display(
    pd.DataFrame([
        {"Split":"Train","Start":0,"EndExclusive":train_end,"Length":train_end},
        {"Split":"Validation","Start":train_end,"EndExclusive":val_end,"Length":val_end-train_end},
        {"Split":"Test","Start":val_end,"EndExclusive":test_end,"Length":test_end-val_end},
    ])
)

print("Exchange path:", EXCHANGE_PATH)
print("Shape:", raw.shape)
print("Columns:", list(raw_df.columns))
print("train_end / val_end / test_end:", train_end, val_end, test_end)


In [ ]:

def prefix_normalize(
    raw_array,
    prefix,
):
    prefix = int(
        prefix
    )

    mean = raw_array[
        :prefix
    ].mean(
        axis=0,
    ).astype(
        np.float32
    )

    std = raw_array[
        :prefix
    ].std(
        axis=0,
        ddof=0,
    ).astype(
        np.float32
    )

    if np.any(
        std
        <= 1e-6
    ):
        raise ValueError(
            f"Degenerate prefix channel at prefix={prefix}."
        )

    z = (
        (
            raw_array
            - mean[
                None,
                :
            ]
        )
        / std[
            None,
            :
        ]
    ).astype(
        np.float32
    )

    return (
        z,
        {
            "Mean":
                mean,
            "Std":
                std,
            "Prefix":
                prefix,
        },
    )


def eval_anchors(
    start,
    end,
    horizon,
    stride=1,
    lookback=RET_SEQ_LEN,
):
    return np.arange(
        max(
            int(
                start
            ),
            int(
                lookback
            ),
        ),
        int(
            end
        )
        - int(
            horizon
        )
        + 1,
        int(
            stride
        ),
        dtype=np.int64,
    )


In [ ]:
# 36F.0 already resolved the repository and imported the model package.
# Here we only verify the state and import training utilities.

assert SEGMOE_REPO is not None
assert (SEGMOE_REPO / "segmoe_forecast/model/TSFT.py").is_file()
assert str(SEGMOE_REPO) in sys.path

from segmoe_forecast.model import TSFTransformer
from segmoe_forecast.model.Config import BaseConfig
from segmoe_forecast.utils import (
    CosineLRDecay,
    EarlyStopping,
    LoadBalancingLoss,
    Trainer,
)

print("Seg-MoE repo:", SEGMOE_REPO)
print("Commit:", SEGMOE_COMMIT)
print("TSFT:", Path(inspect.getfile(TSFTransformer)).resolve())
print("PASS: Seg-MoE repository/configuration preflight.")

pd.DataFrame(
    [
        {
            "Repository": str(SEGMOE_REPO),
            "Commit": SEGMOE_COMMIT,
            "BatchSize": SEGMOE_BATCH_SIZE,
            "RecipeSource": "Predeclared ETTh1-small surrogate; no official Exchange recipe",
            "BF16": SEGMOE_BF16,
        }
    ]
).to_csv(
    ARTIFACT_DIR / "segmoe_repository_and_runtime.csv",
    index=False,
)


In [ ]:
def set_seed(
    seed,
):
    random.seed(
        int(seed)
    )

    np.random.seed(
        int(seed)
    )

    torch.manual_seed(
        int(seed)
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            int(seed)
        )

    torch.backends.cudnn.benchmark = False


def load_torch(
    path,
):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


def segmoe_config():
    cfg = SmallConfig()

    cfg = replace(
        cfg,
        channels=
            n_channels,
        block_size=
            SEGMOE_BLOCK_SIZE,
        patch_width=
            SEGMOE_PATCH_WIDTH,
        width_factor=
            SEGMOE_WIDTH_FACTOR,
        exp_route_dropout=
            0.1,
        exp_route_temperature=
            1.0,
        exp_segment_size=
            SEGMOE_SEGMENT_SIZE,
    )

    return cfg


def build_segmoe_model():
    cfg = segmoe_config()

    model = TSFTransformer.from_config(
        cfg
    ).to(
        DEVICE
    )

    return (
        model,
        cfg,
    )


def direct_seq_len(
    backbone=None,
):
    return SEGMOE_BLOCK_SIZE


def make_direct_batch(
    z,
    anchors,
    backbone_unused,
    horizon,
):
    anchors = np.asarray(
        anchors,
        dtype=np.int64,
    )

    L = SEGMOE_BLOCK_SIZE

    x_idx = (
        anchors[
            :,
            None
        ]
        - L
        + np.arange(
            L
        )[
            None,
            :
        ]
    )

    y_idx = (
        anchors[
            :,
            None
        ]
        + np.arange(
            int(horizon)
        )[
            None,
            :
        ]
    )

    x = z[
        x_idx,
        :
    ].astype(
        np.float32
    )

    y = z[
        y_idx,
        :
    ].astype(
        np.float32
    )

    return (
        x,
        y,
    )


@torch.no_grad()
def forward_direct(
    backbone_unused,
    model,
    x,
    y,
    horizon,
):
    # Existing OOF infrastructure uses [B,T,C].
    x_ltc = torch.from_numpy(
        x
    ).to(
        DEVICE
    )

    y_ltc = torch.from_numpy(
        y
    ).to(
        DEVICE
    )

    # Seg-MoE native layout: [B,C,T].
    x_bct = x_ltc.permute(
        0,
        2,
        1,
    ).contiguous()

    old_n_outputs = int(
        model.n_outputs
    )

    model.n_outputs = int(
        horizon
    )

    try:
        pred_bct = model.forecast(
            x_bct,
            dynamic_window=True,
        )
    finally:
        model.n_outputs = old_n_outputs

    pred_ltc = pred_bct.permute(
        0,
        2,
        1,
    ).contiguous()

    return (
        pred_ltc.float(),
        y_ltc.float(),
        x_ltc.float(),
    )


@torch.no_grad()
def direct_residual_block(
    model,
    z,
    marks_unused,
    anchors,
    horizon,
):
    x, y = make_direct_batch(
        z,
        anchors,
        "SegMoE",
        horizon,
    )

    pred, true, x_t = forward_direct(
        "SegMoE",
        model,
        x,
        y,
        horizon,
    )

    current = x_t[
        :,
        -1:,
        :
    ].float()

    return (
        pred
        - current,
        true
        - current,
    )


class SegMoETrainDataset(
    Dataset
):
    """
    Matches Seg-MoE Dataset_Custom encoder training geometry:
      input  = 512 history points
      target = last 480 history points + next 32 future points
             = 512 points total
    """
    def __init__(
        self,
        z,
        anchors,
    ):
        self.z = z
        self.anchors = np.asarray(
            anchors,
            dtype=np.int64,
        )

    def __len__(
        self,
    ):
        return len(
            self.anchors
        )

    def __getitem__(
        self,
        idx,
    ):
        a = int(
            self.anchors[
                idx
            ]
        )

        x = self.z[
            a
            - SEGMOE_BLOCK_SIZE:
            a,
            :
        ].T

        y = self.z[
            a
            - SEGMOE_LABEL_LEN:
            a
            + SEGMOE_STEP,
            :
        ].T

        return (
            torch.from_numpy(
                np.ascontiguousarray(
                    x,
                    dtype=np.float32,
                )
            ),
            torch.from_numpy(
                np.ascontiguousarray(
                    y,
                    dtype=np.float32,
                )
            ),
            torch.tensor(
                idx,
                dtype=torch.long,
            ),
        )


def segmoe_train_anchors(
    boundary,
):
    return np.arange(
        SEGMOE_BLOCK_SIZE,
        int(boundary)
        - SEGMOE_STEP
        + 1,
        dtype=np.int64,
    )


def segmoe_val_anchors():
    return np.arange(
        train_end,
        val_end
        - SEGMOE_STEP
        + 1,
        dtype=np.int64,
    )


def make_segmoe_loader(
    z,
    anchors,
    shuffle,
    seed,
):
    ds = SegMoETrainDataset(
        z,
        anchors,
    )

    g = torch.Generator()
    g.manual_seed(
        int(seed)
    )

    return DataLoader(
        ds,
        batch_size=
            SEGMOE_BATCH_SIZE,
        shuffle=
            bool(shuffle),
        num_workers=
            0,
        pin_memory=
            torch.cuda.is_available(),
        drop_last=
            False,
        generator=
            g,
    )


def build_segmoe_trainer(
    model,
    train_loader,
    val_loader,
    checkpoint_dir,
    filename,
    do_validation,
    checkpointing,
):
    # v4: reproduce Seg-MoE's parameter-group policy but force
    # torch.optim.AdamW(fused=False) for numerical stability.
    param_dict = {
        name: p
        for name, p
        in model.named_parameters()
        if p.requires_grad
    }

    decay_params = [
        p
        for p in param_dict.values()
        if p.dim() >= 2
    ]

    nodecay_params = [
        p
        for p in param_dict.values()
        if p.dim() < 2
    ]

    optim_groups = [
        {
            "params": decay_params,
            "weight_decay": SEGMOE_WEIGHT_DECAY,
        },
        {
            "params": nodecay_params,
            "weight_decay": 0.0,
        },
    ]

    adamw_kwargs = dict(
        params=optim_groups,
        lr=SEGMOE_MAX_LR,
        betas=(0.9, 0.95),
        eps=1e-10,
    )

    if "fused" in inspect.signature(torch.optim.AdamW).parameters:
        adamw_kwargs["fused"] = False

    optimizer = torch.optim.AdamW(
        **adamw_kwargs
    )

    criterion = nn.HuberLoss(
        reduction="none",
        delta=
            SEGMOE_HUBER_DELTA,
    )

    aux_criterion = LoadBalancingLoss(
        model.config.n_experts,
        model.config.top_k_experts,
        alpha=
            SEGMOE_BAL_ALPHA,
    )

    steps_per_epoch = len(
        train_loader
    )

    max_steps = (
        steps_per_epoch
        * SEGMOE_EPOCHS
    )

    warmup_steps = int(
        max_steps
        * SEGMOE_WARMUP
    )

    scheduler = CosineLRDecay(
        optimizer,
        SEGMOE_MIN_LR,
        SEGMOE_MAX_LR,
        warmup_steps,
        max_steps,
    )

    early_stopping = (
        EarlyStopping(
            patience=
                SEGMOE_PATIENCE,
            min_delta=
                SEGMOE_STOP_MIN,
        )
        if do_validation
        else None
    )

    trainer = Trainer(
        model=
            model,
        device=
            DEVICE,
        train_loader=
            train_loader,
        train_ds_scaler=
            None,
        val_loader=
            val_loader,
        test_loader=
            None,
        criterion=
            criterion,
        optimizer=
            optimizer,
        scheduler=
            scheduler,
        aux_criterion=
            aux_criterion,
        early_stopping=
            early_stopping,
        use_time_features=
            False,
        do_validation=
            bool(do_validation),
        checkpointing=
            bool(checkpointing),
        checkpoint_dir=
            checkpoint_dir,
        filename=
            filename,
        verbose=
            True,
        disable_tqdm=
            True,
    )

    return trainer


In [ ]:
def _router_probs_finite(router_probs):
    if router_probs is None:
        return True

    if not isinstance(router_probs, (tuple, list)):
        return bool(torch.isfinite(router_probs).all())

    for p in router_probs:
        if p is None:
            continue
        if not bool(torch.isfinite(p).all()):
            return False

    return True


def segmoe_precision_smoke_test():
    global SEGMOE_BF16
    global SEGMOE_PRECISION_MODE

    print("=" * 96)
    print("36F SEG-MOE NUMERICAL STABILITY SMOKE TEST")
    print("=" * 96)

    raw_finite = bool(np.isfinite(raw).all())
    z_finite = bool(np.isfinite(z_full).all())

    print("raw finite:", raw_finite)
    print("z_full finite:", z_finite)
    print(
        "z_full range:",
        float(np.min(z_full)),
        "to",
        float(np.max(z_full)),
    )

    if not (raw_finite and z_finite):
        raise FloatingPointError(
            "Exchange data contains NaN/Inf before Seg-MoE training."
        )

    smoke_anchors = segmoe_train_anchors(train_end)[:SEGMOE_BATCH_SIZE]

    smoke_loader = make_segmoe_loader(
        z_full,
        smoke_anchors,
        shuffle=False,
        seed=SEGMOE_SEED,
    )

    data, target, *_ = next(iter(smoke_loader))
    data = data.to(DEVICE)
    target = target.to(DEVICE)

    print("batch data:", tuple(data.shape), data.dtype)
    print("batch target:", tuple(target.shape), target.dtype)
    print(
        "batch finite:",
        bool(torch.isfinite(data).all()),
        bool(torch.isfinite(target).all()),
    )

    criterion = nn.HuberLoss(
        reduction="none",
        delta=SEGMOE_HUBER_DELTA,
    )

    # FP32 must pass.
    set_seed(SEGMOE_SEED)
    fp32_model, _ = build_segmoe_model()
    fp32_model.train()

    fp32_aux = LoadBalancingLoss(
        fp32_model.config.n_experts,
        fp32_model.config.top_k_experts,
        alpha=SEGMOE_BAL_ALPHA,
    )

    with torch.no_grad():
        fp32_logits, fp32_router, *_ = fp32_model(data)
        fp32_task = criterion(fp32_logits, target).mean()
        fp32_aux_loss, _, _ = fp32_aux(
            fp32_router,
            None,
            False,
        )
        fp32_total = fp32_task + fp32_aux_loss

    fp32_ok = bool(
        torch.isfinite(fp32_logits).all()
        and torch.isfinite(fp32_task).all()
        and torch.isfinite(fp32_total).all()
        and _router_probs_finite(fp32_router)
    )

    print("FP32 task loss:", float(fp32_task.detach().cpu()))
    print(
        "FP32 aux loss:",
        float(torch.as_tensor(fp32_aux_loss).detach().cpu()),
    )
    print("FP32 total loss:", float(fp32_total.detach().cpu()))
    print("FP32 router finite:", _router_probs_finite(fp32_router))
    print("FP32 smoke:", "PASS" if fp32_ok else "FAIL")

    del fp32_model, fp32_logits, fp32_router, fp32_task, fp32_aux_loss, fp32_total
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if not fp32_ok:
        raise FloatingPointError(
            "Seg-MoE is non-finite even in FP32. "
            "Do not start the long run."
        )

    # Optional BF16 check.
    if SEGMOE_BF16_REQUESTED and SEGMOE_BF16_SUPPORTED:
        set_seed(SEGMOE_SEED)
        bf16_model, _ = build_segmoe_model()
        bf16_model.train()

        bf16_aux = LoadBalancingLoss(
            bf16_model.config.n_experts,
            bf16_model.config.top_k_experts,
            alpha=SEGMOE_BAL_ALPHA,
        )

        bf16_ok = False
        try:
            with torch.no_grad():
                with torch.amp.autocast(
                    device_type="cuda",
                    dtype=torch.bfloat16,
                ):
                    bf16_logits, bf16_router, *_ = bf16_model(data)
                    bf16_task = criterion(bf16_logits, target).mean()
                    bf16_aux_loss, _, _ = bf16_aux(
                        bf16_router,
                        None,
                        False,
                    )
                    bf16_total = bf16_task + bf16_aux_loss

            bf16_ok = bool(
                torch.isfinite(bf16_logits).all()
                and torch.isfinite(bf16_task).all()
                and torch.isfinite(bf16_total).all()
                and _router_probs_finite(bf16_router)
            )

            print(
                "BF16 task loss:",
                float(bf16_task.float().detach().cpu()),
            )
            print(
                "BF16 aux loss:",
                float(torch.as_tensor(bf16_aux_loss).float().detach().cpu()),
            )
            print(
                "BF16 total loss:",
                float(bf16_total.float().detach().cpu()),
            )
            print("BF16 router finite:", _router_probs_finite(bf16_router))

        except Exception as exc:
            print("BF16 smoke raised:", repr(exc))
            bf16_ok = False

        del bf16_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if bf16_ok:
            SEGMOE_BF16 = True
            print("BF16 smoke: PASS")
        else:
            SEGMOE_BF16 = False
            print(
                "WARNING: BF16 numerical smoke failed. "
                "Falling back to FP32."
            )
    else:
        SEGMOE_BF16 = False
        if SEGMOE_BF16_REQUESTED:
            print(
                "BF16 requested but not supported by the current CUDA device. "
                "Using FP32."
            )
        else:
            print("BF16 explicitly disabled. Using FP32.")

    SEGMOE_PRECISION_MODE = (
        "BF16" if SEGMOE_BF16 else "FP32"
    )

    print("\nFINAL 36F TRAINING PRECISION:", SEGMOE_PRECISION_MODE)
    print("=" * 96)

    return {
        "RawFinite": raw_finite,
        "ZFinite": z_finite,
        "FP32Finite": fp32_ok,
        "BF16Requested": SEGMOE_BF16_REQUESTED,
        "BF16Supported": SEGMOE_BF16_SUPPORTED,
        "FinalPrecision": SEGMOE_PRECISION_MODE,
    }


SEGMOE_SMOKE = segmoe_precision_smoke_test()

pd.DataFrame([SEGMOE_SMOKE]).to_csv(
    ARTIFACT_DIR / "segmoe_numerical_smoke_test.csv",
    index=False,
)


In [ ]:
def _all_gradients_finite(model):
    for p in model.parameters():
        if p.grad is not None and not bool(torch.isfinite(p.grad).all()):
            return False
    return True


def _all_parameters_finite(model):
    for p in model.parameters():
        if not bool(torch.isfinite(p).all()):
            return False
    return True


def build_nonfused_probe_optimizer(model):
    params = {
        name: p
        for name, p in model.named_parameters()
        if p.requires_grad
    }

    groups = [
        {
            "params": [p for p in params.values() if p.dim() >= 2],
            "weight_decay": SEGMOE_WEIGHT_DECAY,
        },
        {
            "params": [p for p in params.values() if p.dim() < 2],
            "weight_decay": 0.0,
        },
    ]

    kwargs = dict(
        params=groups,
        lr=SEGMOE_MAX_LR,
        betas=(0.9, 0.95),
        eps=1e-10,
    )

    if "fused" in inspect.signature(torch.optim.AdamW).parameters:
        kwargs["fused"] = False

    return torch.optim.AdamW(**kwargs)


def segmoe_multistep_stability_test():
    print("=" * 100)
    print("36F MULTI-STEP OPTIMIZER STABILITY TEST")
    print("=" * 100)

    configure_segmoe_safe_backends()

    print("Training precision:", SEGMOE_PRECISION_MODE)
    print("Clip grad:", SEGMOE_CLIP_GRAD)

    set_seed(SEGMOE_SEED)

    model, _ = build_segmoe_model()
    model.train()

    optimizer = build_nonfused_probe_optimizer(model)

    criterion = nn.HuberLoss(
        reduction="none",
        delta=SEGMOE_HUBER_DELTA,
    )

    aux_criterion = LoadBalancingLoss(
        model.config.n_experts,
        model.config.top_k_experts,
        alpha=SEGMOE_BAL_ALPHA,
    )

    train_anchors = segmoe_train_anchors(train_end)
    needed = min(
        len(train_anchors),
        SEGMOE_STABILITY_STEPS * SEGMOE_BATCH_SIZE,
    )

    loader = make_segmoe_loader(
        z_full,
        train_anchors[:needed],
        shuffle=False,
        seed=SEGMOE_SEED,
    )

    full_steps_per_epoch = math.ceil(
        len(train_anchors) / SEGMOE_BATCH_SIZE
    )

    max_steps = full_steps_per_epoch * SEGMOE_EPOCHS
    warmup_steps = int(max_steps * SEGMOE_WARMUP)

    scheduler = CosineLRDecay(
        optimizer,
        SEGMOE_MIN_LR,
        SEGMOE_MAX_LR,
        warmup_steps,
        max_steps,
    )

    rows = []

    for step, batch in enumerate(loader, start=1):
        if step > SEGMOE_STABILITY_STEPS:
            break

        optimizer.zero_grad(set_to_none=True)

        data, target, *_ = batch
        data = data.to(DEVICE)
        target = target.to(DEVICE)

        amp_ctx = (
            torch.amp.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            )
            if SEGMOE_BF16
            else nullcontext()
        )

        with amp_ctx:
            logits, router_probs, *_ = model(data)
            task_loss = criterion(logits, target).mean()
            aux_loss, _, _ = aux_criterion(
                router_probs,
                None,
                False,
            )
            total_loss = task_loss + aux_loss

        forward_ok = bool(
            torch.isfinite(logits).all()
            and torch.isfinite(task_loss).all()
            and torch.isfinite(torch.as_tensor(aux_loss)).all()
            and torch.isfinite(total_loss).all()
            and _router_probs_finite(router_probs)
        )

        if not forward_ok:
            raise FloatingPointError(
                f"36F stability probe: non-finite forward/loss at step {step}."
            )

        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=SEGMOE_CLIP_GRAD,
        )

        grad_ok = _all_gradients_finite(model)
        if not grad_ok:
            raise FloatingPointError(
                f"36F stability probe: non-finite gradient at step {step}."
            )

        lr_before = float(optimizer.param_groups[0]["lr"])
        optimizer.step()

        param_ok = _all_parameters_finite(model)
        if not param_ok:
            raise FloatingPointError(
                f"36F stability probe: non-finite parameter after step {step}."
            )

        scheduler.step()
        lr_after = float(optimizer.param_groups[0]["lr"])

        rows.append({
            "Step": step,
            "Precision": SEGMOE_PRECISION_MODE,
            "TaskLoss": float(task_loss.detach().float().cpu()),
            "AuxLoss": float(torch.as_tensor(aux_loss).detach().float().cpu()),
            "TotalLoss": float(total_loss.detach().float().cpu()),
            "LRBeforeSchedulerStep": lr_before,
            "LRAfterSchedulerStep": lr_after,
            "ForwardFinite": forward_ok,
            "GradFinite": grad_ok,
            "ParamFinite": param_ok,
        })

        if step <= 5 or step % 10 == 0 or step == SEGMOE_STABILITY_STEPS:
            print(
                f"step={step:3d} | "
                f"task={rows[-1]['TaskLoss']:.6f} | "
                f"aux={rows[-1]['AuxLoss']:.6f} | "
                f"total={rows[-1]['TotalLoss']:.6f} | "
                f"lr={lr_before:.3e} -> {lr_after:.3e}"
            )

    probe_df = pd.DataFrame(rows)

    if len(probe_df) < min(SEGMOE_STABILITY_STEPS, len(loader)):
        raise RuntimeError("36F stability probe ended earlier than expected.")

    path = ARTIFACT_DIR / "segmoe_multistep_stability_probe.csv"
    probe_df.to_csv(path, index=False)

    print(f"PASS: {len(probe_df)} optimizer steps remained finite.")
    print("Saved:", path)

    del model, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return probe_df


SEGMOE_STABILITY_PROBE = segmoe_multistep_stability_test()


In [ ]:
FULL_SEGMOE_DIR = (
    EXP36F_ROOT
    / "segmoe_full"
)
FULL_SEGMOE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FULL_SEGMOE_PATH = (
    FULL_SEGMOE_DIR
    / "ExchangeFull8_SegMoE_full.pt"
)

FOLD_SEGMOE_DIR = (
    EXP36F_ROOT
    / "segmoe_fold"
)
FOLD_SEGMOE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def train_or_load_full_segmoe():
    model, cfg = build_segmoe_model()

    if (
        RESUME
        and FULL_SEGMOE_PATH.is_file()
        and not FORCE
    ):
        ckpt = load_torch(
            FULL_SEGMOE_PATH
        )

        if (
            ckpt.get("Protocol")
            != "ExchangeFull8SegMoE36F"
        ):
            raise RuntimeError(
                "Unexpected 36F full Seg-MoE checkpoint protocol."
            )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ],
            strict=True,
        )

        model.eval()

        print(
            "Loaded full Seg-MoE | "
            f"best epoch={ckpt['BestEpoch']} | "
            f"best val={ckpt['BestValLoss']:.6f}"
        )

        return (
            model,
            ckpt,
        )

    set_seed(
        SEGMOE_SEED
    )

    train_loader = make_segmoe_loader(
        z_full,
        segmoe_train_anchors(
            train_end
        ),
        shuffle=True,
        seed=
            SEGMOE_SEED,
    )

    val_loader = make_segmoe_loader(
        z_full,
        segmoe_val_anchors(),
        shuffle=False,
        seed=
            SEGMOE_SEED + 1,
    )

    official_dir = (
        FULL_SEGMOE_DIR
        / "official_trainer"
    )
    official_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    filename = (
        "ExchangeFull8_SegMoE36F"
    )

    trainer = build_segmoe_trainer(
        model,
        train_loader,
        val_loader,
        official_dir,
        filename,
        do_validation=True,
        checkpointing=True,
    )

    t0 = time.time()

    trainer.train(
        SEGMOE_EPOCHS,
        use_bf16=
            SEGMOE_BF16,
        clip_grad=
            SEGMOE_CLIP_GRAD,
        get_moe_metrics=
            True,
    )

    runtime_minutes = (
        time.time()
        - t0
    ) / 60.0

    official_path = (
        official_dir
        / f"{filename}.pth"
    )

    if not official_path.is_file():
        raise FileNotFoundError(
            f"Seg-MoE Trainer did not produce {official_path}"
        )

    best = load_torch(
        official_path
    )

    model.load_state_dict(
        best[
            "model_state_dict"
        ],
        strict=True,
    )

    model.eval()

    best_epoch = int(
        best[
            "epoch"
        ]
    ) + 1

    ckpt = {
        "Protocol":
            "ExchangeFull8SegMoE36F",
        "TrainingPrecision":
            SEGMOE_PRECISION_MODE,
        "OptimizerBackend":
            "AdamW_nonfused",
        "SDPABackend":
            "math_only",
        "Dataset":
            "ExchangeFull8",
        "RecipeSource":
            "Predeclared ETTh1-small surrogate; no official Exchange recipe",
        "Backbone":
            "SegMoE",
        "BestEpoch":
            best_epoch,
        "BestValLoss":
            float(
                best[
                    "best_val_loss"
                ]
            ),
        "RuntimeMinutes":
            runtime_minutes,
        "RepositoryCommit":
            SEGMOE_COMMIT,
        "Recipe": {
            "block_size":
                SEGMOE_BLOCK_SIZE,
            "patch_width":
                SEGMOE_PATCH_WIDTH,
            "width_factor":
                SEGMOE_WIDTH_FACTOR,
            "model_size":
                SEGMOE_MODEL_SIZE,
            "segment_size":
                SEGMOE_SEGMENT_SIZE,
            "clip_grad":
                SEGMOE_CLIP_GRAD,
            "epochs":
                SEGMOE_EPOCHS,
            "max_lr":
                SEGMOE_MAX_LR,
            "min_lr":
                SEGMOE_MIN_LR,
            "weight_decay":
                SEGMOE_WEIGHT_DECAY,
            "warmup":
                SEGMOE_WARMUP,
            "batch_size":
                SEGMOE_BATCH_SIZE,
            "seed":
                SEGMOE_SEED,
            "bf16":
                SEGMOE_BF16,
        },
        "Config":
            asdict(
                model.config
            ),
        "StateDict": {
            k:
                v.detach()
                .cpu()
                .clone()
            for k, v
            in model.state_dict().items()
        },
    }

    torch.save(
        ckpt,
        FULL_SEGMOE_PATH,
    )

    hist = pd.DataFrame({
        "Epoch":
            np.arange(
                1,
                len(
                    best.get(
                        "train_losses",
                        [],
                    )
                )
                + 1,
            ),
        "TrainLoss":
            best.get(
                "train_losses",
                [],
            ),
        "ValLoss":
            (
                best.get(
                    "val_losses",
                    []
                )
                + [np.nan]
                * max(
                    0,
                    len(
                        best.get(
                            "train_losses",
                            [],
                        )
                    )
                    - len(
                        best.get(
                            "val_losses",
                            [],
                        )
                    ),
                )
            )[
                :len(
                    best.get(
                        "train_losses",
                        [],
                    )
                )
            ],
    })

    hist.to_csv(
        FULL_SEGMOE_DIR
        / "training_history_from_best_checkpoint.csv",
        index=False,
    )

    print(
        "Saved full Seg-MoE | "
        f"best epoch={best_epoch} | "
        f"best val={ckpt['BestValLoss']:.6f} | "
        f"runtime={runtime_minutes:.1f} min"
    )

    return (
        model,
        ckpt,
    )


def fold_segmoe_path(
    fold,
):
    return (
        FOLD_SEGMOE_DIR
        / (
            f"ExchangeFull8_SegMoE_F{int(fold)}.pt"
        )
    )


def train_or_load_fold_segmoe(
    z_fold,
    prefix,
    fold,
    fixed_epochs,
):
    path = fold_segmoe_path(
        fold
    )

    model, cfg = build_segmoe_model()

    if (
        RESUME
        and path.is_file()
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        valid = (
            ckpt.get("Protocol")
            == "ExchangeFull8SegMoEOOFDirect36F"
            and int(
                ckpt.get(
                    "Fold",
                    -1,
                )
            )
            == int(fold)
            and int(
                ckpt.get(
                    "Prefix",
                    -1,
                )
            )
            == int(prefix)
            and int(
                ckpt.get(
                    "FixedEpochs",
                    -1,
                )
            )
            == int(fixed_epochs)
        )

        if not valid:
            raise RuntimeError(
                f"Invalid 36F fold Seg-MoE checkpoint: {path}"
            )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ],
            strict=True,
        )

        model.eval()

        print(
            f"Loaded fold Seg-MoE F{fold} | "
            f"prefix={prefix} | "
            f"epochs={fixed_epochs}"
        )

        return (
            model,
            ckpt,
        )

    seed = (
        SEGMOE_SEED
        + 36_000
        + int(fold)
    )

    set_seed(
        seed
    )

    train_loader = make_segmoe_loader(
        z_fold,
        segmoe_train_anchors(
            prefix
        ),
        shuffle=True,
        seed=
            seed,
    )

    fold_log_dir = (
        FOLD_SEGMOE_DIR
        / f"F{fold}_trainer"
    )
    fold_log_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    trainer = build_segmoe_trainer(
        model,
        train_loader,
        val_loader=None,
        checkpoint_dir=
            fold_log_dir,
        filename=
            f"SegMoE_F{fold}",
        do_validation=False,
        checkpointing=False,
    )

    t0 = time.time()

    # Scheduler trajectory still uses the official 20-epoch horizon,
    # but training stops at the validation-selected full-model epoch.
    trainer.train(
        int(
            fixed_epochs
        ),
        use_bf16=
            SEGMOE_BF16,
        clip_grad=
            SEGMOE_CLIP_GRAD,
        get_moe_metrics=
            False,
    )

    runtime_minutes = (
        time.time()
        - t0
    ) / 60.0

    model.eval()

    ckpt = {
        "Protocol":
            "ExchangeFull8SegMoEOOFDirect36F",
        "Dataset":
            "ExchangeFull8",
        "RecipeSource":
            "Predeclared ETTh1-small surrogate; no official Exchange recipe",
        "Backbone":
            "SegMoE",
        "Fold":
            int(fold),
        "Prefix":
            int(prefix),
        "FixedEpochs":
            int(fixed_epochs),
        "Seed":
            int(seed),
        "RuntimeMinutes":
            runtime_minutes,
        "RepositoryCommit":
            SEGMOE_COMMIT,
        "Config":
            asdict(
                model.config
            ),
        "StateDict": {
            k:
                v.detach()
                .cpu()
                .clone()
            for k, v
            in model.state_dict().items()
        },
    }

    torch.save(
        ckpt,
        path,
    )

    print(
        f"Saved fold Seg-MoE F{fold} | "
        f"epochs={fixed_epochs} | "
        f"runtime={runtime_minutes:.1f} min"
    )

    return (
        model,
        ckpt,
    )


In [ ]:

def inv_softplus(
    x,
):
    return math.log(
        math.exp(
            float(
                x
            )
        )
        - 1.0
    )


class PredictivePatchEncoder(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.patch_proj = nn.Linear(
            REP_PATCH_LEN,
            REP_D_MODEL,
        )

        self.pos_embed = nn.Parameter(
            torch.zeros(
                1,
                REP_NUM_PATCHES,
                REP_D_MODEL,
            )
        )

        nn.init.trunc_normal_(
            self.pos_embed,
            std=0.02,
        )

        layer = nn.TransformerEncoderLayer(
            d_model=
                REP_D_MODEL,
            nhead=
                REP_N_HEADS,
            dim_feedforward=
                REP_D_FF,
            dropout=
                REP_DROPOUT,
            activation=
                "gelu",
            batch_first=
                True,
            norm_first=
                True,
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=
                REP_LAYERS,
        )

        self.norm = nn.LayerNorm(
            REP_D_MODEL
        )

        self.proj = nn.Linear(
            REP_D_MODEL,
            REP_DIM,
        )

    def forward(
        self,
        x,
    ):
        p = x.unfold(
            1,
            REP_PATCH_LEN,
            REP_PATCH_STRIDE,
        )

        h = (
            self.patch_proj(
                p
            )
            + self.pos_embed[
                :,
                :p.shape[
                    1
                ],
            ]
        )

        h = self.encoder(
            h
        ).mean(
            dim=1
        )

        h = self.proj(
            self.norm(
                h
            )
        )

        return F.normalize(
            h,
            dim=-1,
            eps=1e-8,
        )


class EmbeddingOnlyRetriever(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.encoder = (
            PredictivePatchEncoder()
        )

        self.raw_gamma = nn.Parameter(
            torch.tensor(
                inv_softplus(
                    1.0
                ),
                dtype=torch.float32,
            )
        )

    @property
    def gamma(
        self,
    ):
        return F.softplus(
            self.raw_gamma
        )

    def encode(
        self,
        x,
    ):
        return self.encoder(
            x
        )


def ret_amp():
    if RETRIEVER_USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return nullcontext()




STRONG_FORECASTER_ROOT = Path(
    "/data/dataset/strong_forecaster"
)

# Exact frozen Exchange retrievers from the earlier experiments.
EXCHANGE_FULL_RETRIEVER_DIR = (
    STRONG_FORECASTER_ROOT
    / "exchange_predictive_representation"
    / "checkpoints"
)

EXCHANGE_FOLD_RETRIEVER_DIR = (
    STRONG_FORECASTER_ROOT
    / "exchange_crossfit_adaptive_gate"
    / "fold_retriever_checkpoints"
)


def full_retriever_path_36f(
    horizon,
):
    path = (
        EXCHANGE_FULL_RETRIEVER_DIR
        / (
            f"Exchange_L96_H{int(horizon)}_"
            "EmbeddingOnly_Listwise_seed0.pt"
        )
    )

    if not path.is_file():
        raise FileNotFoundError(
            f"Missing frozen Exchange full retriever: {path}"
        )

    return path


def fold_retriever_path_36f(
    horizon,
    fold,
):
    path = (
        EXCHANGE_FOLD_RETRIEVER_DIR
        / (
            f"Exchange_H{int(horizon)}_"
            f"Fold{int(fold)}_EmbeddingOnly.pt"
        )
    )

    if not path.is_file():
        raise FileNotFoundError(
            f"Missing frozen Exchange fold retriever: {path}"
        )

    return path


def _extract_retriever_state(
    ckpt,
):
    if not isinstance(
        ckpt,
        dict,
    ):
        return ckpt

    candidate_keys = [
        "StateDict",
        "state_dict",
        "ModelState",
        "model_state_dict",
        "RetrieverState",
        "retriever_state",
        "RetrieverStateDict",
        "retriever_state_dict",
        "Model",
    ]

    for key in candidate_keys:
        value = ckpt.get(
            key
        )

        if isinstance(
            value,
            dict,
        ):
            if (
                len(value) > 0
                and any(
                    torch.is_tensor(v)
                    for v in value.values()
                )
            ):
                return value

    tensor_entries = {
        k: v
        for k, v in ckpt.items()
        if (
            isinstance(k, str)
            and torch.is_tensor(v)
        )
    }

    if tensor_entries:
        return tensor_entries

    raise KeyError(
        "Could not identify retriever state_dict. "
        f"Checkpoint keys: {sorted(map(str, ckpt.keys()))}"
    )


def load_frozen_retriever(
    path,
):
    path = Path(
        path
    )

    if not path.is_file():
        raise FileNotFoundError(
            path
        )

    ckpt = load_torch(
        path
    )

    state = dict(
        _extract_retriever_state(
            ckpt
        )
    )

    legacy_to_current = {
        "encoder.out_norm.weight":
            "encoder.norm.weight",
        "encoder.out_norm.bias":
            "encoder.norm.bias",
        "encoder.out_proj.weight":
            "encoder.proj.weight",
        "encoder.out_proj.bias":
            "encoder.proj.bias",
    }

    for old_key, new_key in legacy_to_current.items():
        if (
            old_key in state
            and new_key not in state
        ):
            state[
                new_key
            ] = state.pop(
                old_key
            )

    for prefix in [
        "module.",
        "model.",
        "retriever.",
    ]:
        keys = list(
            state.keys()
        )

        if (
            keys
            and all(
                k.startswith(prefix)
                for k in keys
            )
        ):
            state = {
                k[len(prefix):]: v
                for k, v in state.items()
            }

    model = EmbeddingOnlyRetriever().to(
        DEVICE
    )

    model.load_state_dict(
        state,
        strict=True,
    )

    model.eval()

    for p in model.parameters():
        p.requires_grad_(
            False
        )

    return (
        model,
        ckpt,
    )


print(
    "Exchange full retriever root:",
    EXCHANGE_FULL_RETRIEVER_DIR,
)

print(
    "Exchange fold retriever root:",
    EXCHANGE_FOLD_RETRIEVER_DIR,
)

for _h in HORIZONS:
    print(
        f"H={_h} full:",
        full_retriever_path_36f(
            _h
        ),
    )

    for _f in range(
        1,
        4,
    ):
        print(
            f"  Fold{_f}:",
            fold_retriever_path_36f(
                _h,
                _f,
            ),
        )


In [ ]:

def retrieval_anchor_batch(
    name,
):
    C = DATA[
        name
    ][
        "n_channels"
    ]

    return max(
        1,
        TARGET_RETRIEVAL_PAIRS
        // C,
    )


def batch_pattern(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    xc = (
        x
        - x.mean(
            axis=-1,
            keepdims=True,
        )
    )

    n = np.linalg.norm(
        xc,
        axis=-1,
        keepdims=True,
    )

    return np.where(
        n > EPS,
        xc
        / np.maximum(
            n,
            EPS,
        ),
        0.0,
    ).astype(
        np.float32
    )


def context7(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    short = max(
        8,
        RET_SEQ_LEN
        // 4,
    )

    m = x.mean(
        axis=-1
    )

    s = (
        x.std(
            axis=-1
        )
        + EPS
    )

    f1 = (
        x[
            ...,
            -1
        ]
        - m
    ) / s

    f2 = (
        x[
            ...,
            -short:
        ].mean(
            axis=-1
        )
        - m
    ) / s

    f3 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            -short
        ]
    ) / s

    f4 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            0
        ]
    ) / s

    df = np.diff(
        x,
        axis=-1,
    )

    ds = np.diff(
        x[
            ...,
            -short:
        ],
        axis=-1,
    )

    f5 = (
        ds.std(
            axis=-1
        )
        + EPS
    ) / (
        df.std(
            axis=-1
        )
        + EPS
    )

    t = np.linspace(
        -1.0,
        1.0,
        RET_SEQ_LEN,
        dtype=np.float32,
    )

    t = (
        t
        - t.mean()
    )

    f6 = (
        np.sum(
            t
            * (
                x
                - m[
                    ...,
                    None
                ]
            ),
            axis=-1,
        )
        / (
            np.sum(
                t
                * t
            )
            + EPS
        )
    ) / s

    a = x[
        ...,
        :-1
    ]

    b = x[
        ...,
        1:
    ]

    a = (
        a
        - a.mean(
            axis=-1,
            keepdims=True,
        )
    )

    b = (
        b
        - b.mean(
            axis=-1,
            keepdims=True,
        )
    )

    f7 = np.sum(
        a
        * b,
        axis=-1,
    ) / (
        np.sqrt(
            np.sum(
                a
                * a,
                axis=-1,
            )
            * np.sum(
                b
                * b,
                axis=-1,
            )
        )
        + EPS
    )

    return np.stack(
        [
            f1,
            f2,
            f3,
            f4,
            f5,
            f6,
            f7,
        ],
        axis=-1,
    ).astype(
        np.float32
    )


def extract_channel(
    z,
    c,
    anchors,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        c,
    ].astype(
        np.float32
    )

    future = z[
        fi,
        c,
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        c,
    ].astype(
        np.float32
    )

    future_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        future_residual,
    )


def build_memory(
    z,
    channels,
    boundary,
    horizon,
):
    memory_anchors = np.arange(
        RET_SEQ_LEN,
        int(
            boundary
        )
        - horizon
        + 1,
        MEMORY_STRIDE,
        dtype=np.int64,
    )

    if len(
        memory_anchors
    ) < TOP_K:
        raise ValueError(
            "Insufficient admissible memory."
        )

    past = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            RET_SEQ_LEN,
        ),
        dtype=np.float32,
    )

    pattern = np.empty_like(
        past
    )

    future = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            horizon,
        ),
        dtype=np.float32,
    )

    for c in range(
        channels
    ):
        p, f = extract_channel(
            z,
            c,
            memory_anchors,
            horizon,
        )

        past[
            c
        ] = p

        pattern[
            c
        ] = batch_pattern(
            p
        )

        future[
            c
        ] = f

    return {
        "anchors":
            memory_anchors,
        "past":
            past,
        "pattern":
            pattern,
        "future":
            future,
        "M":
            len(
                memory_anchors
            ),
        "boundary":
            int(
                boundary
            ),
    }


def query_pairs(
    z,
    anchors,
    channels,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    channels = np.asarray(
        channels,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    future = z[
        fi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        channels,
    ].astype(
        np.float32
    )

    true_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        batch_pattern(
            past
        ),
        context7(
            past
        ),
        true_residual,
    )


In [ ]:

@torch.no_grad()
def encode_np(
    model,
    x,
    chunk=512,
):
    parts = []

    for i in range(
        0,
        len(
            x
        ),
        chunk,
    ):
        t = torch.from_numpy(
            x[
                i:
                i+chunk
            ]
        ).to(
            DEVICE
        )

        with ret_amp():
            e = model.encode(
                t
            ).float()

        parts.append(
            e.cpu()
        )

        del (
            t,
            e,
        )

    return torch.cat(
        parts,
        dim=0,
    ).numpy().astype(
        np.float32
    )


def memory_embedding_path(
    name,
    horizon,
    tag,
):
    return (
        SHARED_MEMORY_EMB_DIR
        / (
            f"{name}_H{horizon}_"
            f"{tag}_emb.npy"
        )
    )


@torch.no_grad()
def memory_gpu_cached(
    name,
    horizon,
    tag,
    model,
    memory,
    channels,
):
    path = memory_embedding_path(
        name,
        horizon,
        tag,
    )

    expected = (
        channels,
        memory[
            "M"
        ],
        REP_DIM,
    )

    emb_np = None

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        candidate = np.load(
            path,
            mmap_mode=None,
        )

        if (
            tuple(
                candidate.shape
            )
            == expected
        ):
            emb_np = candidate.astype(
                np.float32,
                copy=False,
            )

            print(
                "Loaded memory embedding:",
                path.name,
            )

    if emb_np is None:
        emb_np = np.empty(
            expected,
            dtype=np.float32,
        )

        print(
            "Building memory embedding:",
            path.name,
            expected,
        )

        for c in range(
            channels
        ):
            emb_np[
                c
            ] = encode_np(
                model,
                memory[
                    "past"
                ][
                    c
                ],
            )

            if (
                c == 0
                or (
                    c + 1
                )
                % 50
                == 0
                or (
                    c + 1
                    == channels
                )
            ):
                print(
                    f"  channel "
                    f"{c+1}/{channels}"
                )

        np.save(
            path,
            emb_np,
        )

    return {
        "emb":
            torch.from_numpy(
                emb_np
            ).to(
                DEVICE
            ),
        "pattern":
            torch.from_numpy(
                memory[
                    "pattern"
                ]
            ).to(
                DEVICE
            ),
        "future":
            torch.from_numpy(
                memory[
                    "future"
                ]
            ).to(
                DEVICE
            ),
    }


In [ ]:

@torch.no_grad()
def retrieve(
    model,
    memory_gpu_obj,
    z,
    anchors,
    channels,
    horizon,
):
    (
        past,
        pattern,
        ctx,
        true,
    ) = query_pairs(
        z,
        anchors,
        channels,
        horizon,
    )

    past_t = torch.from_numpy(
        past
    ).to(
        DEVICE
    )

    pattern_t = torch.from_numpy(
        pattern
    ).to(
        DEVICE
    )

    with ret_amp():
        qemb = model.encode(
            past_t
        )

    qemb = qemb.float()

    memb = memory_gpu_obj[
        "emb"
    ][
        channels
    ]

    sim = torch.bmm(
        qemb[
            :,
            None,
            :
        ],
        memb.transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    score = (
        model.gamma
        * sim
    )

    idx = torch.topk(
        score,
        TOP_K,
        dim=1,
    ).indices

    row = torch.arange(
        len(
            channels
        ),
        device=DEVICE,
    )[
        :,
        None
    ]

    pfull = torch.bmm(
        pattern_t[
            :,
            None,
            :
        ],
        memory_gpu_obj[
            "pattern"
        ][
            channels
        ].transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    return {
        "score":
            score[
                row,
                idx
            ],
        "sim":
            sim[
                row,
                idx
            ],
        "pattern":
            pfull[
                row,
                idx
            ],
        "cand":
            memory_gpu_obj[
                "future"
            ][
                channels[
                    :,
                    None
                ],
                idx,
            ],
        "ctx":
            torch.from_numpy(
                ctx
            ).to(
                DEVICE
            ),
        "true":
            torch.from_numpy(
                true
            ).to(
                DEVICE
            ),
    }


In [ ]:

class CrossFitAdaptiveGate(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                GATE_DIM,
                64,
            ),
            nn.LayerNorm(
                64
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                64,
                32,
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                32,
                1,
            ),
        )

        nn.init.normal_(
            self.net[
                -1
            ].weight,
            mean=0.0,
            std=1e-3,
        )

        nn.init.constant_(
            self.net[
                -1
            ].bias,
            math.log(
                0.1
                / 0.9
            ),
        )

    def forward(
        self,
        x,
    ):
        return torch.sigmoid(
            self.net(
                x
            ).squeeze(
                -1
            )
        )


def score_entropy(
    s,
):
    p = torch.softmax(
        s,
        dim=1,
    )

    return (
        -(
            p
            * torch.log(
                p.clamp_min(
                    1e-8
                )
            )
        ).sum(
            dim=1
        )
        / math.log(
            TOP_K
        )
    )


def feature_cosine(
    a,
    b,
):
    return (
        (
            a
            * b
        ).sum(
            dim=1
        )
        / (
            torch.sqrt(
                (
                    a
                    * a
                ).sum(
                    dim=1
                )
                + 1e-8
            )
            * torch.sqrt(
                (
                    b
                    * b
                ).sum(
                    dim=1
                )
                + 1e-8
            )
        )
    )


def gate_features(
    r,
    retrieval,
    direct,
):
    s = r[
        "score"
    ]

    sim = r[
        "sim"
    ]

    pattern = r[
        "pattern"
    ]

    sorted_s = torch.sort(
        s,
        dim=1,
        descending=True,
    ).values

    cand_std = r[
        "cand"
    ].std(
        dim=1,
        unbiased=False,
    )

    disp_rms = torch.sqrt(
        (
            cand_std
            * cand_std
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disp_mean = cand_std.mean(
        dim=1
    )

    direct_rms = torch.sqrt(
        (
            direct
            * direct
        ).mean(
            dim=1
        )
        + 1e-8
    )

    retrieval_rms = torch.sqrt(
        (
            retrieval
            * retrieval
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disagreement = (
        retrieval
        - direct
    )

    disagreement_rms = torch.sqrt(
        (
            disagreement
            * disagreement
        ).mean(
            dim=1
        )
        + 1e-8
    )

    relative_disagreement = (
        disagreement_rms
        / (
            direct_rms
            + retrieval_rms
            + 1e-6
        )
    )

    scalars = torch.stack(
        [
            s.mean(
                dim=1
            ),
            s.std(
                dim=1,
                unbiased=False,
            ),
            s.max(
                dim=1
            ).values,
            sorted_s[
                :,
                0
            ]
            - sorted_s[
                :,
                1
            ],
            s.max(
                dim=1
            ).values
            - s.mean(
                dim=1
            ),
            score_entropy(
                s
            ),
            sim.mean(
                dim=1
            ),
            sim.std(
                dim=1,
                unbiased=False,
            ),
            sim.max(
                dim=1
            ).values,
            pattern.mean(
                dim=1
            ),
            pattern.std(
                dim=1,
                unbiased=False,
            ),
            pattern.max(
                dim=1
            ).values,
            disp_rms,
            disp_mean,
            direct_rms,
            retrieval_rms,
            disagreement_rms,
            relative_disagreement,
            feature_cosine(
                direct,
                retrieval,
            ),
        ],
        dim=1,
    )

    out = torch.cat(
        [
            r[
                "ctx"
            ],
            scalars,
        ],
        dim=1,
    )

    if out.shape[
        1
    ] != GATE_DIM:
        raise RuntimeError(
            f"Gate feature dimension mismatch: "
            f"{out.shape}"
        )

    return out


def abc_terms(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    return torch.stack(
        [
            (
                e
                * e
            ).mean(
                dim=1
            ),
            (
                e
                * delta
            ).mean(
                dim=1
            ),
            (
                delta
                * delta
            ).mean(
                dim=1
            ),
        ],
        dim=1,
    )


In [ ]:

@torch.no_grad()
def collect_gate_data(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    z,
    anchors,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    ret_block = retrieval_anchor_batch(
        name
    )

    direct_block = DIRECT_ANCHOR_BLOCK[
        CURRENT_BACKBONE
    ]

    features = []
    abcs = []
    anchors_out = []
    channels_out = []

    for outer in range(
        0,
        len(
            anchors
        ),
        direct_block,
    ):
        a_big = anchors[
            outer:
            outer+direct_block
        ]

        direct_big, true_big = (
            direct_residual_block(
                direct_model,
                z,
                data[
                    "marks"
                ],
                a_big,
                horizon,
            )
        )

        # [A, H, C] -> [A, C, H]
        direct_big = direct_big.permute(
            0,
            2,
            1,
        ).contiguous()

        true_big = true_big.permute(
            0,
            2,
            1,
        ).contiguous()

        for inner in range(
            0,
            len(
                a_big
            ),
            ret_block,
        ):
            a = a_big[
                inner:
                inner+ret_block
            ]

            A = len(
                a
            )

            pair_anchor = np.repeat(
                a,
                C,
            )

            pair_channel = np.tile(
                np.arange(
                    C,
                    dtype=np.int64,
                ),
                A,
            )

            r = retrieve(
                retriever,
                memory_gpu_obj,
                z,
                pair_anchor,
                pair_channel,
                horizon,
            )

            retrieval = r[
                "cand"
            ].mean(
                dim=1
            )

            d = direct_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            t = true_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            # Strong consistency check:
            # direct true residual must match retrieval true residual.
            max_true_diff = float(
                (
                    t
                    - r[
                        "true"
                    ]
                ).abs().max()
            )

            if (
                max_true_diff
                > 2e-5
            ):
                raise RuntimeError(
                    f"True residual mismatch: "
                    f"{max_true_diff}"
                )

            feat = gate_features(
                r,
                retrieval,
                d,
            )

            abc = abc_terms(
                d,
                retrieval,
                t,
            )

            features.append(
                feat.cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            abcs.append(
                abc.cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            anchors_out.append(
                pair_anchor
            )

            channels_out.append(
                pair_channel
            )

            del (
                r,
                retrieval,
                d,
                t,
                feat,
                abc,
            )

        del (
            direct_big,
            true_big,
        )

    return {
        "feature":
            np.concatenate(
                features,
                axis=0,
            ),
        "abc":
            np.concatenate(
                abcs,
                axis=0,
            ),
        "anchor":
            np.concatenate(
                anchors_out,
                axis=0,
            ),
        "channel":
            np.concatenate(
                channels_out,
                axis=0,
            ),
    }


In [ ]:

def fit_feature_scaler(
    x,
):
    median = np.median(
        x,
        axis=0,
    ).astype(
        np.float32
    )

    q25 = np.percentile(
        x,
        25,
        axis=0,
    )

    q75 = np.percentile(
        x,
        75,
        axis=0,
    )

    iqr = (
        q75
        - q25
    ).astype(
        np.float32
    )

    iqr = np.where(
        iqr < 1e-5,
        1.0,
        iqr,
    ).astype(
        np.float32
    )

    return (
        median,
        iqr,
    )


def scale_features(
    x,
    median,
    iqr,
):
    return np.clip(
        (
            x
            - median
        )
        / iqr,
        -8.0,
        8.0,
    ).astype(
        np.float32
    )


def gate_loss(
    alpha,
    abc,
):
    return (
        abc[
            :,
            0
        ]
        + 2.0
        * alpha
        * abc[
            :,
            1
        ]
        + alpha
        * alpha
        * abc[
            :,
            2
        ]
    ).mean()


def gate_checkpoint_path(
    name,
    horizon,
):
    return (
        DIRS[
            "gate"
        ]
        / (
            f"{name}_H{horizon}_"
            "gate.pt"
        )
    )


def train_gate_epoch(
    model,
    optimizer,
    x,
    abc,
    rng,
):
    model.train()

    order = rng.permutation(
        len(
            x
        )
    )

    losses = []

    for i in range(
        0,
        len(
            order
        ),
        GATE_BATCH,
    ):
        ids = order[
            i:
            i+GATE_BATCH
        ]

        xt = torch.from_numpy(
            x[
                ids
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                ids
            ]
        ).to(
            DEVICE
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        alpha = model(
            xt
        )

        loss = gate_loss(
            alpha,
            at,
        )

        loss.backward()
        optimizer.step()

        losses.append(
            float(
                loss.item()
            )
        )

        del (
            xt,
            at,
            alpha,
            loss,
        )

    return float(
        np.mean(
            losses
        )
    )


@torch.no_grad()
def evaluate_gate(
    model,
    x,
    abc,
):
    model.eval()

    total = 0.0
    n = 0
    alpha_sum = 0.0

    for i in range(
        0,
        len(
            x
        ),
        GATE_BATCH,
    ):
        xt = torch.from_numpy(
            x[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        alpha = model(
            xt
        )

        each = (
            at[
                :,
                0
            ]
            + 2.0
            * alpha
            * at[
                :,
                1
            ]
            + alpha
            * alpha
            * at[
                :,
                2
            ]
        )

        total += float(
            each.sum()
        )

        n += len(
            alpha
        )

        alpha_sum += float(
            alpha.sum()
        )

        del (
            xt,
            at,
            alpha,
            each,
        )

    return (
        total
        / n,
        alpha_sum
        / n,
    )


def train_crossfit_gate(
    name,
    horizon,
    oof_x,
    oof_abc,
    val_x,
    val_abc,
):
    path = gate_checkpoint_path(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model = CrossFitAdaptiveGate().to(
            DEVICE
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            "Loaded gate:",
            path.name,
        )

        return (
            model,
            ckpt,
        )

    median, iqr = fit_feature_scaler(
        oof_x
    )

    train_x = scale_features(
        oof_x,
        median,
        iqr,
    )

    valid_x = scale_features(
        val_x,
        median,
        iqr,
    )

    seed = (
        CROSSFIT_SEED
        + horizon
        * 3000
        + sum(
            map(
                ord,
                name,
            )
        )
    )

    set_seed(
        seed
    )

    model = CrossFitAdaptiveGate().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    rng = np.random.default_rng(
        seed
        + 1
    )

    best = float(
        "inf"
    )

    best_epoch = -1
    wait = 0
    history = []

    for epoch in range(
        1,
        GATE_MAX_EPOCHS
        + 1,
    ):
        train_mse = train_gate_epoch(
            model,
            optimizer,
            train_x,
            oof_abc,
            rng,
        )

        val_mse, mean_alpha = evaluate_gate(
            model,
            valid_x,
            val_abc,
        )

        history.append({
            "Epoch":
                epoch,
            "OOFTrainMSE":
                train_mse,
            "ValMSE":
                val_mse,
            "ValMeanAlpha":
                mean_alpha,
        })

        if (
            val_mse
            < best
            - 1e-10
        ):
            best = val_mse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        print(
            f"Gate {name:11s} H={horizon:3d} "
            f"ep={epoch:02d} "
            f"OOF={train_mse:.6f} "
            f"val={val_mse:.6f} "
            f"alpha={mean_alpha:.3f} "
            f"best={best:.6f}@{best_epoch}"
        )

        if (
            wait
            >= GATE_PATIENCE
        ):
            break

    # Reinitialize and fit only on OOF for the
    # validation-selected number of epochs.
    set_seed(
        seed
    )

    final = CrossFitAdaptiveGate().to(
        DEVICE
    )

    final_opt = torch.optim.AdamW(
        final.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    final_rng = np.random.default_rng(
        seed
        + 2
    )

    for _ in range(
        best_epoch
    ):
        train_gate_epoch(
            final,
            final_opt,
            train_x,
            oof_abc,
            final_rng,
        )

    final.eval()

    ckpt = {
        "BestEpoch":
            best_epoch,
        "BestValMSE":
            best,
        "FeatureMedian":
            median,
        "FeatureIQR":
            iqr,
        "StateDict": {
            k:
                v.detach()
                .cpu()
                .clone()
            for k, v
            in final.state_dict().items()
        },
    }

    torch.save(
        ckpt,
        path,
    )

    pd.DataFrame(
        history
    ).to_csv(
        DIRS[
            "history"
        ]
        / (
            f"{name}_H{horizon}_"
            "gate_history.csv"
        ),
        index=False,
    )

    return (
        final,
        ckpt,
    )


def mse_scalar(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = float(
        alpha
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_scalar(
    abc,
):
    rows = []

    best_alpha = None
    best_mse = float(
        "inf"
    )

    for alpha in ALPHA_GRID:
        mse = mse_scalar(
            abc,
            alpha,
        )

        rows.append({
            "Alpha":
                float(
                    alpha
                ),
            "MSE":
                mse,
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_alpha = float(
                alpha
            )

    return (
        best_alpha,
        pd.DataFrame(
            rows
        ),
    )


@torch.no_grad()
def gate_alpha(
    model,
    ckpt,
    x,
):
    sx = scale_features(
        x,
        ckpt[
            "FeatureMedian"
        ],
        ckpt[
            "FeatureIQR"
        ],
    )

    outputs = []

    for i in range(
        0,
        len(
            sx
        ),
        GATE_BATCH,
    ):
        t = torch.from_numpy(
            sx[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        outputs.append(
            model(
                t
            ).cpu()
            .numpy()
        )

        del t

    return np.concatenate(
        outputs
    ).astype(
        np.float32
    )


def mse_pair(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = np.asarray(
        alpha,
        dtype=np.float64,
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_lambda(
    abc,
    gate_alpha_values,
    scalar_alpha,
):
    rows = []

    best_lambda = None
    best_mse = float(
        "inf"
    )

    for lmb in LAMBDA_GRID:
        alpha = (
            (
                1.0
                - float(
                    lmb
                )
            )
            * scalar_alpha
            + float(
                lmb
            )
            * gate_alpha_values
        )

        mse = mse_pair(
            abc,
            alpha,
        )

        rows.append({
            "Lambda":
                float(
                    lmb
                ),
            "MSE":
                mse,
            "MeanAlpha":
                float(
                    alpha.mean()
                ),
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_lambda = float(
                lmb
            )

    return (
        best_lambda,
        pd.DataFrame(
            rows
        ),
    )


In [ ]:

def empty_stat():
    return {
        "sse":
            0.0,
        "sae":
            0.0,
        "n":
            0,
    }


def update_stat(
    stat,
    pred,
    true,
):
    e = (
        pred
        - true
    )

    stat[
        "sse"
    ] += float(
        (
            e
            * e
        ).sum()
    )

    stat[
        "sae"
    ] += float(
        e.abs().sum()
    )

    stat[
        "n"
    ] += e.numel()


def finish_stat(
    stat,
):
    return (
        stat[
            "sse"
        ]
        / stat[
            "n"
        ],
        stat[
            "sae"
        ]
        / stat[
            "n"
        ],
    )


def oracle_alpha_and_prediction(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    alpha = torch.clamp(
        -(
            e
            * delta
        ).sum(
            dim=1
        )
        / (
            (
                delta
                * delta
            ).sum(
                dim=1
            )
            + 1e-8
        ),
        0.0,
        1.0,
    )

    pred = (
        direct
        + alpha[
            :,
            None
        ]
        * delta
    )

    return (
        alpha,
        pred,
    )


@torch.no_grad()
def test_evaluate(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    gate,
    gate_ckpt,
    scalar_alpha,
    shrink_lambda,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    direct_block = DIRECT_ANCHOR_BLOCK[
        CURRENT_BACKBONE
    ]

    ret_block = retrieval_anchor_batch(
        name
    )

    anchors = eval_anchors(
        data[
            "val_end"
        ],
        data[
            "test_end"
        ],
        horizon,
        stride=1,
    )

    keys = [
        "Direct",
        "Retrieval",
        "Scalar",
        "RawAdaptive",
        "ShrinkAdaptive",
        "Oracle",
    ]

    stats = {
        key:
            empty_stat()
        for key in keys
    }

    anchor_mse = {
        key:
            []
        for key in keys
    }

    channel_sse_direct = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_sse_shrink = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_count = np.zeros(
        C,
        dtype=np.int64,
    )

    raw_alpha_sum = 0.0
    shrink_alpha_sum = 0.0
    oracle_alpha_sum = 0.0
    oracle_positive = 0
    n_pairs = 0

    processed = 0

    for outer in range(
        0,
        len(
            anchors
        ),
        direct_block,
    ):
        a_big = anchors[
            outer:
            outer+direct_block
        ]

        direct_big, true_big = (
            direct_residual_block(
                direct_model,
                data[
                    "z"
                ],
                data[
                    "marks"
                ],
                a_big,
                horizon,
            )
        )

        direct_big = direct_big.permute(
            0,
            2,
            1,
        ).contiguous()

        true_big = true_big.permute(
            0,
            2,
            1,
        ).contiguous()

        # anchor-level MSE accumulators within this direct block
        block_anchor_sums = {
            key:
                torch.zeros(
                    len(
                        a_big
                    ),
                    device=DEVICE,
                    dtype=torch.float64,
                )
            for key in keys
        }

        for inner in range(
            0,
            len(
                a_big
            ),
            ret_block,
        ):
            a = a_big[
                inner:
                inner+ret_block
            ]

            A = len(
                a
            )

            pair_anchor = np.repeat(
                a,
                C,
            )

            pair_channel = np.tile(
                np.arange(
                    C,
                    dtype=np.int64,
                ),
                A,
            )

            r = retrieve(
                retriever,
                memory_gpu_obj,
                data[
                    "z"
                ],
                pair_anchor,
                pair_channel,
                horizon,
            )

            retrieval = r[
                "cand"
            ].mean(
                dim=1
            )

            direct = direct_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            true = true_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            scalar = (
                direct
                + scalar_alpha
                * (
                    retrieval
                    - direct
                )
            )

            features = gate_features(
                r,
                retrieval,
                direct,
            ).cpu().numpy().astype(
                np.float32
            )

            scaled = scale_features(
                features,
                gate_ckpt[
                    "FeatureMedian"
                ],
                gate_ckpt[
                    "FeatureIQR"
                ],
            )

            gate_alpha_values = gate(
                torch.from_numpy(
                    scaled
                ).to(
                    DEVICE
                )
            )

            shrink_alpha_values = (
                (
                    1.0
                    - shrink_lambda
                )
                * scalar_alpha
                + shrink_lambda
                * gate_alpha_values
            )

            raw_adaptive = (
                direct
                + gate_alpha_values[
                    :,
                    None
                ]
                * (
                    retrieval
                    - direct
                )
            )

            shrink_adaptive = (
                direct
                + shrink_alpha_values[
                    :,
                    None
                ]
                * (
                    retrieval
                    - direct
                )
            )

            (
                oracle_alpha,
                oracle,
            ) = oracle_alpha_and_prediction(
                direct,
                retrieval,
                true,
            )

            predictions = {
                "Direct":
                    direct,
                "Retrieval":
                    retrieval,
                "Scalar":
                    scalar,
                "RawAdaptive":
                    raw_adaptive,
                "ShrinkAdaptive":
                    shrink_adaptive,
                "Oracle":
                    oracle,
            }

            for key, pred in predictions.items():
                update_stat(
                    stats[
                        key
                    ],
                    pred,
                    true,
                )

                per_anchor = (
                    (
                        (
                            pred
                            - true
                        )
                        ** 2
                    )
                    .reshape(
                        A,
                        C,
                        horizon,
                    )
                    .mean(
                        dim=(
                            1,
                            2,
                        )
                    )
                    .double()
                )

                block_anchor_sums[
                    key
                ][
                    inner:
                    inner+A
                ] = per_anchor

            direct_e2 = (
                (
                    direct
                    - true
                )
                ** 2
            ).reshape(
                A,
                C,
                horizon,
            )

            shrink_e2 = (
                (
                    shrink_adaptive
                    - true
                )
                ** 2
            ).reshape(
                A,
                C,
                horizon,
            )

            channel_sse_direct += (
                direct_e2.sum(
                    dim=(
                        0,
                        2,
                    )
                ).cpu()
                .numpy()
            )

            channel_sse_shrink += (
                shrink_e2.sum(
                    dim=(
                        0,
                        2,
                    )
                ).cpu()
                .numpy()
            )

            channel_count += (
                A
                * horizon
            )

            raw_alpha_sum += float(
                gate_alpha_values.sum()
            )

            shrink_alpha_sum += float(
                shrink_alpha_values.sum()
            )

            oracle_alpha_sum += float(
                oracle_alpha.sum()
            )

            oracle_positive += int(
                (
                    oracle_alpha
                    > 0.01
                ).sum()
            )

            n_pairs += len(
                gate_alpha_values
            )

            del (
                r,
                retrieval,
                direct,
                true,
                scalar,
                features,
                scaled,
                gate_alpha_values,
                shrink_alpha_values,
                raw_adaptive,
                shrink_adaptive,
                oracle_alpha,
                oracle,
                predictions,
                direct_e2,
                shrink_e2,
            )

        for key in keys:
            anchor_mse[
                key
            ].extend(
                block_anchor_sums[
                    key
                ].cpu()
                .numpy()
                .astype(
                    np.float32
                )
                .tolist()
            )

        processed += len(
            a_big
        )

        if (
            processed
            == len(
                a_big
            )
            or processed
            % 500
            < len(
                a_big
            )
            or processed
            == len(
                anchors
            )
        ):
            print(
                f"  test anchors "
                f"{processed}/{len(anchors)}"
            )

        del (
            direct_big,
            true_big,
            block_anchor_sums,
        )

    return {
        "anchors":
            anchors,
        "metrics": {
            key:
                finish_stat(
                    value
                )
            for key, value
            in stats.items()
        },
        "anchor_mse": {
            key:
                np.asarray(
                    value,
                    dtype=np.float32,
                )
            for key, value
            in anchor_mse.items()
        },
        "channel_direct_mse":
            channel_sse_direct
            / channel_count,
        "channel_shrink_mse":
            channel_sse_shrink
            / channel_count,
        "raw_mean_alpha":
            raw_alpha_sum
            / n_pairs,
        "shrink_mean_alpha":
            shrink_alpha_sum
            / n_pairs,
        "oracle_mean_alpha":
            oracle_alpha_sum
            / n_pairs,
        "oracle_positive_fraction":
            oracle_positive
            / n_pairs,
    }


In [ ]:

def moving_block_bootstrap(
    difference,
    n_boot=5000,
    block=24,
    seed=222222,
):
    x = np.asarray(
        difference,
        dtype=np.float64,
    )

    n = len(
        x
    )

    L = min(
        block,
        n,
    )

    rng = np.random.default_rng(
        seed
    )

    n_blocks = int(
        np.ceil(
            n
            / L
        )
    )

    max_start = max(
        1,
        n
        - L
        + 1,
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):
        starts = rng.integers(
            0,
            max_start,
            size=n_blocks,
        )

        sample = np.concatenate(
            [
                x[
                    s:
                    s+L
                ]
                for s in starts
            ]
        )[
            :n
        ]

        boot[
            b
        ] = sample.mean()

    return {
        "MeanImprovement":
            float(
                x.mean()
            ),
        "CI_Low":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),
        "CI_High":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),
    }


In [ ]:
audit_rows = []

for horizon in HORIZONS:
    full_path = full_retriever_path_36f(horizon)

    try:
        m, _ = load_frozen_retriever(full_path)
        full_ok, full_err = True, None
        del m
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as exc:
        full_ok, full_err = False, repr(exc)

    audit_rows.append({
        "Horizon": int(horizon),
        "Tag": "Full",
        "Exists": full_path.is_file(),
        "LoadableStrictly": full_ok,
        "Error": full_err,
        "Path": str(full_path),
    })

    for fold in range(1, 4):
        path = fold_retriever_path_36f(horizon, fold)

        try:
            m, _ = load_frozen_retriever(path)
            ok, err = True, None
            del m
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as exc:
            ok, err = False, repr(exc)

        audit_rows.append({
            "Horizon": int(horizon),
            "Tag": f"Fold{fold}",
            "Exists": path.is_file(),
            "LoadableStrictly": ok,
            "Error": err,
            "Path": str(path),
        })

retriever_audit = pd.DataFrame(audit_rows)
display(retriever_audit)

if not bool(retriever_audit["LoadableStrictly"].all()):
    bad = retriever_audit[
        ~retriever_audit["LoadableStrictly"]
    ]

    raise RuntimeError(
        "36F requires all frozen Exchange full/fold retrievers.\n"
        + bad.to_string(index=False)
    )

print("PASS: all 4 full + 12 fold Exchange retrievers from predictive-representation/crossfit checkpoints load strictly.")


def _seed_exchange_cache(filename):
    dst = SHARED_MEMORY_EMB_DIR / filename

    if dst.is_file():
        return "already"

    hits = [
        Path(p)
        for p in STRONG_FORECASTER_ROOT.rglob(filename)
        if EXP36F_ROOT not in Path(p).parents
    ]

    if not hits:
        return "missing"

    hits = sorted(
        hits,
        key=lambda p: (
            len(p.parts),
            str(p),
        ),
    )

    src = hits[0]

    try:
        os.link(src, dst)
        return "linked"
    except Exception:
        shutil.copy2(src, dst)
        return "copied"


cache_rows = []

for horizon in HORIZONS:
    for fold, (p0, p1) in enumerate(FOLDS, start=1):
        prefix = int(p0 * train_end)

        filename = (
            f"Exchange_H{int(horizon)}_"
            f"F{fold}_prefix{prefix}_emb.npy"
        )

        cache_rows.append({
            "Horizon": int(horizon),
            "Tag": f"F{fold}",
            "File": filename,
            "Status": _seed_exchange_cache(filename),
        })

    for tag in [
        "validation_train_memory",
        "test_trainval_memory",
    ]:
        filename = (
            f"Exchange_H{int(horizon)}_"
            f"{tag}_emb.npy"
        )

        cache_rows.append({
            "Horizon": int(horizon),
            "Tag": tag,
            "File": filename,
            "Status": _seed_exchange_cache(filename),
        })

cache_audit = pd.DataFrame(cache_rows)
display(cache_audit)

print(
    "Exchange memory-cache preflight complete. "
    "Missing caches are safe and will be built on demand."
)


In [ ]:
full_segmoe, full_segmoe_ckpt = (
    train_or_load_full_segmoe()
)

SEGMOE_FIXED_FOLD_EPOCHS = int(
    full_segmoe_ckpt[
        "BestEpoch"
    ]
)

if SEGMOE_FIXED_FOLD_EPOCHS < 1:
    raise RuntimeError(
        "Invalid full Seg-MoE BestEpoch."
    )

print(
    "Fixed fold epoch budget:",
    SEGMOE_FIXED_FOLD_EPOCHS,
)

fold_models = {}

for fold, (
    p0,
    p1,
) in enumerate(
    FOLDS,
    start=1,
):
    prefix = int(
        p0
        * train_end
    )

    z_fold, _ = prefix_normalize(
        raw,
        prefix,
    )

    model, ckpt = (
        train_or_load_fold_segmoe(
            z_fold,
            prefix,
            fold,
            SEGMOE_FIXED_FOLD_EPOCHS,
        )
    )

    fold_models[
        fold
    ] = model

print(
    "PASS: full Seg-MoE + 3 fold Seg-MoE models are ready."
)


In [ ]:
OOF36F_DIR = (
    EXP36F_ROOT
    / "oof"
)
OOF36F_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

VALIDATION36F_DIR = (
    EXP36F_ROOT
    / "validation"
)
VALIDATION36F_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def legacy_cache_identity_36f(horizon):
    return "Exchange"


def oof_path_36f(
    horizon,
    fold,
):
    return (
        OOF36F_DIR
        / (
            f"ExchangeFull8_SegMoE_"
            f"H{int(horizon)}_F{int(fold)}_oof.npz"
        )
    )


def build_oof_fold_36f(
    horizon,
    fold,
    p0,
    p1,
):
    horizon = int(
        horizon
    )

    fold = int(
        fold
    )

    out_path = oof_path_36f(
        horizon,
        fold,
    )

    if (
        RESUME
        and out_path.is_file()
        and not FORCE
    ):
        obj = np.load(
            out_path
        )

        out = {
            key:
                obj[
                    key
                ]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

        uc = np.unique(
            out[
                "channel"
            ]
        )

        if (
            len(uc) == 8
            and int(
                uc.min()
            ) == 0
            and int(
                uc.max()
            ) == 8
        ):
            print(
                f"Loaded 36F OOF H={horizon} F{fold}: "
                f"{len(out['abc']):,} pairs"
            )

            return out

    prefix = int(
        p0
        * train_end
    )

    oof_end = int(
        p1
        * train_end
    )

    z_fold, scaler = prefix_normalize(
        raw,
        prefix,
    )

    direct_model = fold_models[
        fold
    ]

    retriever_path = fold_retriever_path_36f(
        horizon,
        fold,
    )

    retriever, _ = load_frozen_retriever(
        retriever_path
    )

    memory = build_memory(
        z_fold,
        n_channels,
        prefix,
        horizon,
    )

    memory_gpu_obj = memory_gpu_cached(
        legacy_cache_identity_36f(
            horizon
        ),
        horizon,
        f"F{fold}_prefix{prefix}",
        retriever,
        memory,
        n_channels,
    )

    anchors = eval_anchors(
        prefix,
        oof_end,
        horizon,
        stride=
            OOF_ANCHOR_STRIDE,
        lookback=
            SEGMOE_BLOCK_SIZE,
    )

    print(
        f"36F OOF H={horizon} F{fold} | "
        f"anchors={len(anchors):,} | "
        f"pairs={len(anchors)*n_channels:,}"
    )

    out = collect_gate_data(
        DATA[
            "Exchange"
        ],
        horizon,
        direct_model,
        retriever,
        memory_gpu_obj,
        z_fold,
        anchors,
    )

    np.savez_compressed(
        out_path,
        **out,
    )

    del (
        retriever,
        memory,
        memory_gpu_obj,
        z_fold,
        scaler,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


def validation_path_36f(
    horizon,
):
    return (
        VALIDATION36F_DIR
        / (
            f"ExchangeFull8_SegMoE_"
            f"H{int(horizon)}_validation.npz"
        )
    )


def build_validation_36f(
    horizon,
    full_direct,
    full_retriever,
):
    horizon = int(
        horizon
    )

    path = validation_path_36f(
        horizon
    )

    if (
        RESUME
        and path.is_file()
        and not FORCE
    ):
        obj = np.load(
            path
        )

        out = {
            key:
                obj[
                    key
                ]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

        if (
            len(
                np.unique(
                    out[
                        "channel"
                    ]
                )
            )
            == 8
        ):
            print(
                f"Loaded 36F validation H={horizon}."
            )

            return out

    memory = build_memory(
        z_full,
        n_channels,
        train_end,
        horizon,
    )

    memory_gpu_obj = memory_gpu_cached(
        legacy_cache_identity_36f(
            horizon
        ),
        horizon,
        "validation_train_memory",
        full_retriever,
        memory,
        n_channels,
    )

    anchors = eval_anchors(
        train_end,
        val_end,
        horizon,
        stride=1,
        lookback=
            SEGMOE_BLOCK_SIZE,
    )

    out = collect_gate_data(
        DATA[
            "Exchange"
        ],
        horizon,
        full_direct,
        full_retriever,
        memory_gpu_obj,
        z_full,
        anchors,
    )

    np.savez_compressed(
        path,
        **out,
    )

    del (
        memory,
        memory_gpu_obj,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


In [ ]:
existing_summary = (
    pd.read_csv(
        SUMMARY_36F
    )
    if SUMMARY_36F.is_file()
    else pd.DataFrame()
)

existing_bootstrap = (
    pd.read_csv(
        BOOTSTRAP_36F
    )
    if BOOTSTRAP_36F.is_file()
    else pd.DataFrame()
)

summary_rows = (
    existing_summary.to_dict(
        "records"
    )
    if len(
        existing_summary
    )
    else []
)

bootstrap_rows = (
    existing_bootstrap.to_dict(
        "records"
    )
    if len(
        existing_bootstrap
    )
    else []
)

completed = set(
    int(
        x
    )
    for x in (
        existing_summary[
            "Horizon"
        ].tolist()
        if (
            len(
                existing_summary
            )
            and "Horizon"
            in existing_summary
        )
        else []
    )
)

for horizon in HORIZONS:
    horizon = int(
        horizon
    )

    if (
        RESUME
        and horizon in completed
        and not FORCE
    ):
        print(
            f"Skipping completed H={horizon}."
        )
        continue

    t_h = time.time()

    print(
        "\n"
        + "=" * 108
    )
    print(
        f"36F START — Exchange full-8 Seg-MoE H={horizon}"
    )
    print(
        "=" * 108
    )

    # -------------------------------------------------------------
    # 1. Three chronological OOF folds.
    # -------------------------------------------------------------
    oof_parts = []

    for fold, (
        p0,
        p1,
    ) in enumerate(
        FOLDS,
        start=1,
    ):
        out = build_oof_fold_36f(
            horizon,
            fold,
            p0,
            p1,
        )

        oof_parts.append(
            out
        )

    oof = {
        key:
            np.concatenate(
                [
                    part[
                        key
                    ]
                    for part
                    in oof_parts
                ],
                axis=0,
            )
        for key in [
            "feature",
            "abc",
            "anchor",
            "channel",
        ]
    }

    print(
        f"H={horizon} combined OOF | "
        f"pairs={len(oof['abc']):,} | "
        f"anchors={len(np.unique(oof['anchor'])):,} | "
        f"channels={len(np.unique(oof['channel']))}"
    )

    # -------------------------------------------------------------
    # 2. Full frozen retriever.
    # -------------------------------------------------------------
    full_retriever_path = (
        full_retriever_path_36f(
            horizon
        )
    )

    full_retriever, full_ret_ckpt = (
        load_frozen_retriever(
            full_retriever_path
        )
    )

    # -------------------------------------------------------------
    # 3. Standard validation.
    # -------------------------------------------------------------
    val = build_validation_36f(
        horizon,
        full_segmoe,
        full_retriever,
    )

    # -------------------------------------------------------------
    # 4. OOF-trained gate.
    # -------------------------------------------------------------
    gate_name = (
        f"ExchangeFull8_36F_SegMoE_H{horizon}"
    )

    gate, gate_ckpt = (
        train_crossfit_gate(
            gate_name,
            horizon,
            oof[
                "feature"
            ],
            oof[
                "abc"
            ],
            val[
                "feature"
            ],
            val[
                "abc"
            ],
        )
    )

    # -------------------------------------------------------------
    # 5. Validation-only scalar / shrinkage.
    # -------------------------------------------------------------
    scalar_alpha, scalar_table = (
        choose_scalar(
            val[
                "abc"
            ]
        )
    )

    calibration_dir = (
        EXP36F_ROOT
        / "calibration"
    )
    calibration_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    scalar_table.to_csv(
        calibration_dir
        / (
            f"H{horizon}_"
            "validation_scalar_alpha.csv"
        ),
        index=False,
    )

    val_gate_alpha = gate_alpha(
        gate,
        gate_ckpt,
        val[
            "feature"
        ],
    )

    shrink_lambda, lambda_table = (
        choose_lambda(
            val[
                "abc"
            ],
            val_gate_alpha,
            scalar_alpha,
        )
    )

    lambda_table.to_csv(
        calibration_dir
        / (
            f"H{horizon}_"
            "validation_shrink_lambda.csv"
        ),
        index=False,
    )

    val_direct_mse = mse_scalar(
        val[
            "abc"
        ],
        0.0,
    )

    val_final_alpha = (
        (
            1.0
            - shrink_lambda
        )
        * scalar_alpha
        + shrink_lambda
        * val_gate_alpha
    )

    val_final_mse = mse_pair(
        val[
            "abc"
        ],
        val_final_alpha,
    )

    print(
        f"36F validation H={horizon} | "
        f"Direct={val_direct_mse:.6f} | "
        f"Final={val_final_mse:.6f} | "
        f"scalar={scalar_alpha:.2f} | "
        f"lambda={shrink_lambda:.2f} | "
        f"gain="
        f"{100*(val_direct_mse-val_final_mse)/val_direct_mse:+.3f}%"
    )

    # -------------------------------------------------------------
    # 6. Final test, train+validation memory.
    # -------------------------------------------------------------
    test_memory = build_memory(
        z_full,
        n_channels,
        val_end,
        horizon,
    )

    test_memory_gpu = memory_gpu_cached(
        legacy_cache_identity_36f(
            horizon
        ),
        horizon,
        "test_trainval_memory",
        full_retriever,
        test_memory,
        n_channels,
    )

    test = test_evaluate(
        DATA[
            "Exchange"
        ],
        horizon,
        full_segmoe,
        full_retriever,
        test_memory_gpu,
        gate,
        gate_ckpt,
        scalar_alpha,
        shrink_lambda,
    )

    metrics = test[
        "metrics"
    ]

    direct_mse, direct_mae = (
        metrics[
            "Direct"
        ]
    )

    retrieval_mse, retrieval_mae = (
        metrics[
            "Retrieval"
        ]
    )

    scalar_mse, scalar_mae = (
        metrics[
            "Scalar"
        ]
    )

    raw_mse, raw_mae = (
        metrics[
            "RawAdaptive"
        ]
    )

    final_mse, final_mae = (
        metrics[
            "ShrinkAdaptive"
        ]
    )

    oracle_mse, oracle_mae = (
        metrics[
            "Oracle"
        ]
    )

    diff = (
        test[
            "anchor_mse"
        ][
            "Direct"
        ]
        - test[
            "anchor_mse"
        ][
            "ShrinkAdaptive"
        ]
    )

    boot = moving_block_bootstrap(
        diff,
        n_boot=
            BOOTSTRAP_REPLICATES,
        block=
            BOOTSTRAP_BLOCK_LEN,
        seed=
            BOOTSTRAP_SEED
            + horizon,
    )

    channel_df = pd.DataFrame({
        "Channel":
            np.arange(
                n_channels
            ),
        "Direct_MSE":
            test[
                "channel_direct_mse"
            ],
        "OOF_Memory_MSE":
            test[
                "channel_shrink_mse"
            ],
    })

    channel_df[
        "Gain_pct"
    ] = (
        100.0
        * (
            channel_df[
                "Direct_MSE"
            ]
            - channel_df[
                "OOF_Memory_MSE"
            ]
        )
        / channel_df[
            "Direct_MSE"
        ]
    )

    channel_dir = (
        EXP36F_ROOT
        / "channel_results"
    )
    channel_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    channel_df.to_csv(
        channel_dir
        / f"H{horizon}.csv",
        index=False,
    )

    improved_channel_fraction = float(
        (
            channel_df[
                "OOF_Memory_MSE"
            ]
            < channel_df[
                "Direct_MSE"
            ]
        ).mean()
    )

    runtime_minutes = (
        time.time()
        - t_h
    ) / 60.0

    row = {
        "Backbone":
            "SegMoE",
        "Dataset":
            "ExchangeFull8",
        "Horizon":
            horizon,
        "Channels":
            n_channels,
        "Protocol":
            "ExchangeFull8SegMoEOOFReranker36F",
        "TrainingPrecision":
            SEGMOE_PRECISION_MODE,
        "OptimizerBackend":
            "AdamW_nonfused",
        "SDPABackend":
            "math_only",
        "Direct_MSE":
            direct_mse,
        "Final_MSE":
            final_mse,
        "Direct_MAE":
            direct_mae,
        "Final_MAE":
            final_mae,
        "Retrieval_MSE":
            retrieval_mse,
        "Scalar_MSE":
            scalar_mse,
        "RawAdaptive_MSE":
            raw_mse,
        "Oracle_MSE":
            oracle_mse,
        "ScalarAlpha":
            scalar_alpha,
        "ShrinkLambda":
            shrink_lambda,
        "RawMeanAlpha":
            test[
                "raw_mean_alpha"
            ],
        "FinalMeanAlpha":
            test[
                "shrink_mean_alpha"
            ],
        "Validation_Direct_MSE":
            val_direct_mse,
        "Validation_Final_MSE":
            val_final_mse,
        "Validation_Gain_pct":
            100.0
            * (
                val_direct_mse
                - val_final_mse
            )
            / val_direct_mse,
        "MSEGain_pct":
            100.0
            * (
                direct_mse
                - final_mse
            )
            / direct_mse,
        "MAEGain_pct":
            100.0
            * (
                direct_mae
                - final_mae
            )
            / direct_mae,
        "ImprovedChannelFraction":
            improved_channel_fraction,
        "OracleHeadroom_pct":
            100.0
            * (
                direct_mse
                - oracle_mse
            )
            / direct_mse,
        "GateBestEpoch":
            gate_ckpt[
                "BestEpoch"
            ],
        "SegMoEBestEpoch":
            full_segmoe_ckpt[
                "BestEpoch"
            ],
        "SegMoECommit":
            SEGMOE_COMMIT,
        "TestAnchors":
            len(
                test[
                    "anchors"
                ]
            ),
        "RuntimeMinutes":
            runtime_minutes,
    }

    bootstrap_row = {
        "Backbone":
            "SegMoE",
        "Dataset":
            "ExchangeFull8",
        "Horizon":
            horizon,
        "Comparison":
            "Direct-OOFCrossFitMemory",
        "MeanImprovement":
            boot[
                "MeanImprovement"
            ],
        "CI_Low":
            boot[
                "CI_Low"
            ],
        "CI_High":
            boot[
                "CI_High"
            ],
        "SignificantPositive":
            bool(
                boot[
                    "CI_Low"
                ]
                > 0
            ),
        "SignificantNegative":
            bool(
                boot[
                    "CI_High"
                ]
                < 0
            ),
        "BlockLen":
            BOOTSTRAP_BLOCK_LEN,
        "Replicates":
            BOOTSTRAP_REPLICATES,
    }

    summary_rows = [
        r
        for r in summary_rows
        if int(
            r.get(
                "Horizon",
                -1,
            )
        )
        != horizon
    ]

    bootstrap_rows = [
        r
        for r in bootstrap_rows
        if int(
            r.get(
                "Horizon",
                -1,
            )
        )
        != horizon
    ]

    summary_rows.append(
        row
    )

    bootstrap_rows.append(
        bootstrap_row
    )

    pd.DataFrame(
        summary_rows
    ).sort_values(
        "Horizon"
    ).to_csv(
        SUMMARY_36F,
        index=False,
    )

    pd.DataFrame(
        bootstrap_rows
    ).sort_values(
        "Horizon"
    ).to_csv(
        BOOTSTRAP_36F,
        index=False,
    )

    np.savez_compressed(
        EXP36F_ROOT
        / f"H{horizon}_anchor_mse.npz",
        Anchors=
            test[
                "anchors"
            ],
        Direct=
            test[
                "anchor_mse"
            ][
                "Direct"
            ],
        Retrieval=
            test[
                "anchor_mse"
            ][
                "Retrieval"
            ],
        Scalar=
            test[
                "anchor_mse"
            ][
                "Scalar"
            ],
        RawAdaptive=
            test[
                "anchor_mse"
            ][
                "RawAdaptive"
            ],
        Final=
            test[
                "anchor_mse"
            ][
                "ShrinkAdaptive"
            ],
        Oracle=
            test[
                "anchor_mse"
            ][
                "Oracle"
            ],
    )

    print(
        f"\n36F H={horizon} RESULT | "
        f"Direct={direct_mse:.6f} | "
        f"OOF Final={final_mse:.6f} | "
        f"Gain={row['MSEGain_pct']:+.3f}% | "
        f"CI=[{boot['CI_Low']:.6f}, {boot['CI_High']:.6f}] | "
        f"Sig+={boot['CI_Low'] > 0} | "
        f"Sig-={boot['CI_High'] < 0} | "
        f"ImprovedCh={100*improved_channel_fraction:.1f}% | "
        f"Runtime={runtime_minutes:.1f} min"
    )

    del (
        full_retriever,
        test_memory,
        test_memory_gpu,
        gate,
        oof,
        oof_parts,
        val,
        test,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(
    "\nExperiment 36F main loop finished."
)


In [ ]:
s = pd.read_csv(SUMMARY_36F)
b = pd.read_csv(BOOTSTRAP_36F)

if len(s) != 4:
    raise RuntimeError(
        f"36F incomplete: expected 4 horizons, found {len(s)}."
    )

final = (
    s.merge(
        b[
            [
                "Horizon",
                "CI_Low",
                "CI_High",
                "SignificantPositive",
                "SignificantNegative",
            ]
        ],
        on="Horizon",
        how="left",
    )
    .sort_values("Horizon")
)

display(
    final[
        [
            "Horizon",
            "Direct_MSE",
            "Final_MSE",
            "MSEGain_pct",
            "Direct_MAE",
            "Final_MAE",
            "MAEGain_pct",
            "ScalarAlpha",
            "ShrinkLambda",
            "FinalMeanAlpha",
            "ImprovedChannelFraction",
            "OracleHeadroom_pct",
            "CI_Low",
            "CI_High",
            "SignificantPositive",
            "SignificantNegative",
            "SegMoEBestEpoch",
            "RuntimeMinutes",
        ]
    ]
)

print("=" * 112)
print("EXPERIMENT 36F — EXCHANGE FULL8 SEG-MOE + OOF HISTORICAL MEMORY")
print("=" * 112)

wins = int((final["MSEGain_pct"] > 0).sum())
losses = int((final["MSEGain_pct"] < 0).sum())
ties = int((final["MSEGain_pct"].abs() < 1e-12).sum())
sig_wins = int(final["SignificantPositive"].fillna(False).sum())
sig_losses = int(final["SignificantNegative"].fillna(False).sum())

print("MSE wins:", wins, "/ 4")
print("MSE losses:", losses, "/ 4")
print("Exact ties:", ties, "/ 4")
print("Significant wins:", sig_wins, "/ 4")
print("Significant losses:", sig_losses, "/ 4")
print("Mean MSE gain:", f"{final['MSEGain_pct'].mean():+.3f}%")
print("Mean MAE gain:", f"{final['MAEGain_pct'].mean():+.3f}%")
print(
    "Mean improved-channel fraction:",
    f"{100 * final['ImprovedChannelFraction'].mean():.1f}%",
)
print("Mean final alpha:", f"{final['FinalMeanAlpha'].mean():.4f}")
print("Mean oracle headroom:", f"{final['OracleHeadroom_pct'].mean():+.3f}%")

print("\nPer-horizon:")

for _, r in final.iterrows():
    print(
        f"H={int(r['Horizon']):3d} | "
        f"Direct={r['Direct_MSE']:.6f} | "
        f"Final={r['Final_MSE']:.6f} | "
        f"Gain={r['MSEGain_pct']:+.3f}% | "
        f"CI=[{r['CI_Low']:.6f}, {r['CI_High']:.6f}] | "
        f"ImprovedCh={100*r['ImprovedChannelFraction']:.1f}% | "
        f"alpha={r['FinalMeanAlpha']:.4f} | "
        f"oracle={r['OracleHeadroom_pct']:+.3f}% | "
        f"Sig+={bool(r['SignificantPositive'])} | "
        f"Sig-={bool(r['SignificantNegative'])}"
    )

print("\nExchange interpretation checkpoint:")

mean_gain = float(final["MSEGain_pct"].mean())
mean_alpha = float(final["FinalMeanAlpha"].mean())
mean_oracle = float(final["OracleHeadroom_pct"].mean())

if mean_gain > 0.5:
    print(
        "- Clear positive Seg-MoE gain. This is stronger than the prior near-neutral "
        "3-backbone behavior and should be analyzed as possible backbone-specific complementarity."
    )
elif mean_gain > 0:
    print(
        "- Small positive aggregate gain: consistent with Exchange as a near-neutral / "
        "candidate-global regime."
    )
elif mean_gain > -0.25:
    print(
        "- Essentially neutral aggregate effect: also consistent with Exchange calibration "
        "mostly shutting historical memory off."
    )
else:
    print(
        "- Negative aggregate effect. Inspect validation-to-test trust shift before "
        "interpreting this as a dataset-level failure."
    )

if mean_alpha < 0.01:
    print(
        "- Calibration nearly shuts off historical memory, matching the expected "
        "candidate-global / limited incremental-utility behavior."
    )

if mean_oracle > 5.0 and mean_gain <= 0.25:
    print(
        "- Oracle headroom remains non-trivial despite weak realized gain: "
        "retrieval/integration remains the bottleneck rather than complete absence of useful histories."
    )

print(
    "\nRecipe caveat: Seg-MoE has no official Exchange recipe. "
    "36F predeclares the ETTh1-small Table-8 recipe as a low-dimensional surrogate "
    "and does not tune it after seeing Exchange test results."
)

final_path = (
    EXP36F_ROOT
    / "segmoe_all4_horizons_results_with_bootstrap.csv"
)

final.to_csv(final_path, index=False)

print("\nSaved:", final_path)
